# Student Retention Analytics: Results and Discussion
**Project:** Student Retention Early-Warning (Philippine Higher Education) · Eskwelabs Innovation Fellowship, Data Innovation Track · **Deliverable 3**
**Builds on:** my whitepaper v17 (*A Two-Layer Synthetic Data Generator for Studying Student Retention Early-Warning Systems in Philippine Higher Education*) and the generator notebook I wrote for Deliverable 2 (`synthetic-data-generator/student-retention-data-generator.ipynb`, v1.0.1). "WP" below means my whitepaper.

> ⚠️ **Everything here is synthetic.** Every number in this notebook describes the behaviour of a generative model that is calibrated to cited *aggregate* statistics. None of it measures any real Philippine student population, and none of it says what any real institution should do. The only real numbers are the calibration inputs tagged `CALIBRATED` in the generator's registry.

### What this notebook does
I use the Deliverable 2 dataset the way a Student Success Analyst would, and I use the hidden ground truth *only* to grade the analysis. The same ground truth is what makes this possible: I know the support programme cuts the dropout hazard by exactly 30 %, I know which students are genuinely vulnerable, and I know the full assignment rule.

| Sec. | Question | Data allowed |
|---|---|---|
| 1 | Setup and guardrails | none |
| 2 | Load the exported dataset and check its integrity | exported tables |
| 3 | Copy the generator engine (verbatim) and prove it reproduces the dataset | engine + exported tables |
| 4 | **Is the generator valid?** Re-derive the checks from the CSVs | exported tables (+ latent for two labelled checks) |
| 5 | **What does the data show?** Descriptive analytics | exported tables |
| 6 | **Can attrition be predicted before it happens?** Early-warning model | exported tables, prior terms only |
| 7 | **Does good prediction tell me whether support works?** The actionability gap | exported tables |
| 8 | **Does the support programme work?** Causal analysis against the known 30 % | exported tables (+ replicate populations) |
| 9 | **Who is flagged, who is missed, what do false flags cost?** | exported tables; latent audit in Sec. 9.2 only |
| 10 | **How robust is all of this?** Sensitivity to the `ASSUMED` parameters | engine (new populations) |
| 11 | Recommendations | none |
| 12 | Limitations, open flags and the D3 assumptions log | none |
| 13 | Reproducibility | none |

### Two rules I kept throughout
1. **The analysis sees only what an analyst would see.** Every number in Sec. 5 to Sec. 9.1 is computed from the eight exported CSV files. The latent state (baseline vulnerability, shocks, hazard) is used in exactly three clearly labelled places: the validation checks in Sec. 4.4 and Sec. 4.6, the ceiling and policy-value calculations in Sec. 6.4 and Sec. 9.2, and the new-population sensitivity runs in Sec. 10. In each of them it grades the analysis. It never feeds it.
2. **Every claim is about the model, not the world.** Where a result depends on an `ASSUMED` choice, I say so and Sec. 10 tests it.

### Method references
* **Law (2015), *Simulation Modeling and Analysis*:** verification and validation of a simulation model, common random numbers, independent replications (Sec. 3, Sec. 4, Sec. 8, Sec. 10).
* **James, Witten, Hastie & Tibshirani (2013), *An Introduction to Statistical Learning*:** cross-validation and the evaluation of classifiers (Sec. 6).
* **Gelman, Hill & Vehtari (2020), *Regression and Other Stories*:** treatment-effect estimation, ignorability and overlap (Sec. 7, Sec. 8).
* **Peters, Janzing & Schölkopf (2017), *Elements of Causal Inference*:** the causal graph, backdoor adjustment and why adjusting for the wrong set fails (Sec. 8).
* **Lopez-Muñoz, Gonzalez Serrano & Sanchez-Ferreira (2026), *JITE: Research* 25, https://doi.org/10.28945/5776:** a systematic review of dropout-prediction work, whose practitioner recommendation is that models should be decision-support tools and not stand-alone solutions. That is the framing for Sec. 7 and Sec. 11.

### Key results (all synthetic; each is traced to the section that supports it)
1. **The generator holds up when I re-derive its checks from the CSV files (Sec. 3, Sec. 4).** The copied engine regenerates all eight tables bit for bit (SHA-256) and the verbatim calibration loop returns the recorded (δ, γ0). Seven of seven re-derived scorecard rows are identical to what the generator reported.
2. **Early warning works, but only modestly (Sec. 6).** Out of fold, boosted trees reach AUC 0.655 (SD 0.005 across folds) against 0.567 for the signal-count rule an institution already has. Reviewing the top 10 % of student-terms captures 21.4 % of dropouts (rule: 15.3 %). Even a predictor that knew every student's hidden baseline vulnerability would reach only 0.683, and one that knew the true hazard 0.722. And 19.4 % of all dropouts happen in term 1, before any prior-term record exists.
3. **A good predictor does not tell me whether the programme works (Sec. 7).** The raw comparison says supported students drop out 58 % *more* often. Stratifying on the model's own risk score gives a 7 % reduction (95 % CI 3.0 % to 11.7 %), which misses the truth. Stratifying on the assignment signals gives 32.2 % (24.3 % to 39.3 %) against a true 30 %.
4. **The choice of estimator matters more than the amount of data (Sec. 8).** Mantel–Haenszel and Poisson recover roughly 30 %, with an SD of about 4 points across 20 populations. Inverse-probability weighting swings from −47 % to +72 % across the same populations, because the 15,293 supported student-terms carry an effective sample size of only 43. Adjusting for only some of the referral signals also fails.
5. **Most dropouts happen where the programme is not looking (Sec. 9).** 63.7 % of dropout events fall in terms with a clean record and no support. At equal capacity, ranking by the boosted-tree model would avert an expected 940 dropouts against 796 for the actual rule (+18 %), and still only 66 % of what an unattainable oracle would.
6. **These conclusions survive what I varied (Sec. 10).** Across 11 scenarios, each recalibrated to the same tier targets, Mantel–Haenszel stays near 30 %. IPW only becomes stable once all signal cells contain both arms.

## 1 · Setup and guardrails
Run settings only. Model parameters of the generator live in its registry (Sec. 3); the choices I add for this deliverable live in the D3 registry below and carry the same `LOCKED` / `CALIBRATED` / `ASSUMED` tags, so nothing I introduce is untraceable.

* `FAST_MODE`: a smoke-test switch that cuts the replicate counts and the sensitivity grid. Results in fast mode are **not** comparable with the generator's scorecard.
* `RUN_SENSITIVITY`: set to `False` to skip Sec. 10, the slowest section (several minutes).

Expected run time in full mode is roughly 10 to 15 minutes on a laptop, almost all of it in Sec. 4.5 (replicate populations) and Sec. 10 (sensitivity).

In [ ]:
import hashlib, json, platform, sys, time, warnings
from contextlib import contextmanager
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy, statsmodels, sklearn
import statsmodels.api as sm
from IPython.display import display, Markdown
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40)

CONFIG = {"FAST_MODE": False, "RUN_SENSITIVITY": True}
FAST = CONFIG["FAST_MODE"]

HERE = Path.cwd() if Path.cwd().name == "results-and-discussion" else Path.cwd() / "results-and-discussion"
DATA = HERE.parent / "synthetic-data-generator" / "output"
FIG = HERE / "figures"; FIG.mkdir(parents=True, exist_ok=True)
assert DATA.is_dir(), f"Cannot find the exported dataset at {DATA}. Run this notebook from the repository root or from results-and-discussion/."

SYNTHETIC_BANNER = ("SYNTHETIC DATA: produced by a generative model. Not real students; "
                    "not a measurement of any real population.")
FIG_TAG = "SYNTHETIC DATA: model behaviour, not real students"
def banner(): print("⚠", SYNTHETIC_BANNER)
def savefig(fig, name, title):
    fig.suptitle(f"{title}   [{FIG_TAG}]", fontsize=10, y=1.0)
    fig.tight_layout()
    fig.savefig(FIG / name, dpi=130, bbox_inches="tight")

# ---- D3 registry: every choice I add for this deliverable carries exactly one tag -----------------------
A3 = {}
def reg3(name, value, tag, basis):
    assert tag in {"LOCKED", "CALIBRATED", "ASSUMED"} and name not in A3
    A3[name] = {"value": value, "tag": tag, "basis": basis}
def Q(name): return A3[name]["value"]

reg3("n_replicates", 3 if FAST else 20, "LOCKED", "WP Table 5 requires at least 20 independent populations (3 in FAST_MODE; not comparable)")
reg3("n_replicates_sens", 2 if FAST else 10, "ASSUMED", "Run-time budget for the sensitivity grid; the SD across 10 runs is itself noisy, so I read it as an order of magnitude")
reg3("n_bootstrap", 200 if FAST else 1000, "ASSUMED", "Student-level bootstrap draws for the naive-estimate interval (same as the generator's default)")
reg3("cv_folds", 5, "ASSUMED", "Grouped (by student) k-fold cross-validation (James et al. 2013)")
reg3("cv_seed", 193, "ASSUMED", "Seed for the fold assignment; one fixed split used by every model so comparisons are paired")
reg3("topk_shares", (0.05, 0.10, 0.20), "ASSUMED", "Review-capacity cut-offs for top-k capture (5 %, 10 %, 20 % of student-terms)")
reg3("hgb_params", dict(max_iter=500, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=200, l2_regularization=1.0,
                        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=193),
     "ASSUMED", "Fixed, deliberately not tuned; early stopping on an inner validation split (James et al. 2013)")
reg3("logit_C", 1.0, "ASSUMED", "Default L2 strength for the discrete-time logistic hazard")
reg3("analyst_knows_thresholds", True, "ASSUMED", "The analyst knows the referral thresholds (grade < 60, attendance < 0.75, any hold) because the institution set them; they are generator parameters tau_g and tau_a")
reg3("min_arm", 100, "ASSUMED", "A signal cell 'contains both arms' if each arm has at least 100 student-terms (same threshold as the generator's positivity table)")
reg3("sens_calib_delta_bracket", (0.0, 4.0), "ASSUMED", "First-try bracket for the tier shift (the generator's own); widened automatically only if the target is not bracketed")
print("D3 registry:", len(A3), "entries |", "FAST_MODE =", FAST)

## 2 · Load the exported dataset and check its integrity
The analysis starts from the eight CSV files in `synthetic-data-generator/output/`, exactly as a downstream user would. Before using them I check three things: the files match the SHA-256 hashes and row counts that the generator recorded in `metadata.json`, no latent column is present, and the keys are intact.

A hash mismatch does not stop the notebook. Hashes of CSV text can differ across platforms (for example through line-ending conversion when a repository is checked out on Windows), so I report the mismatch and carry on. The row counts and the statistical checks in Sec. 4 still apply.

In [ ]:
TABLE_NAMES = ["Regions", "Students", "Enrollments", "CourseGrades", "AttendanceRecords",
               "FinancialHolds", "SupportInterventions", "PersistenceOutcomes"]
META = json.loads((DATA / "metadata.json").read_text())
TB = {n: pd.read_csv(DATA / f"{n}.csv") for n in TABLE_NAMES}
SEED, N, T = META["master_seed"], META["n_students"], 8

integ = pd.DataFrame([{"table": n, "rows": len(TB[n]), "rows match metadata": len(TB[n]) == META["rows"][n],
                       "file SHA-256 matches metadata": hashlib.sha256((DATA / f"{n}.csv").read_bytes()).hexdigest() == META["sha256"][n]}
                      for n in TABLE_NAMES])
display(integ.style.hide(axis="index"))
print(f"Generator {META['generator_version']} | spec {META['spec_version']} | N = {N:,} | seed = {SEED}")
print(f"Calibrated: delta = {META['calibrated']['delta']:.6f}, gamma0 = {META['calibrated']['gamma0']:.6f}")
if not integ["file SHA-256 matches metadata"].all():
    print("NOTE: at least one file hash differs from metadata.json (often a line-ending effect). Continuing; Sec. 3 tests the data against the engine statistically.")

LATENT_NAMES = {"baseline_vulnerability", "shock", "hazard", "vulnerability"}
assert not any(LATENT_NAMES & set(df.columns) for df in TB.values()), "latent column found in an export!"
PANEL6 = ["Enrollments", "CourseGrades", "AttendanceRecords", "FinancialHolds", "SupportInterventions", "PersistenceOutcomes"]
assert all(not TB[n].duplicated(["student_id", "term"]).any() for n in PANEL6)
assert all(TB[n].student_id.isin(TB["Students"].student_id).all() for n in PANEL6)
assert len({len(TB[n]) for n in PANEL6}) == 1
print("✔ no latent column; keys unique; every panel row maps to a student; the six panel tables have identical row counts")
banner()

## 3 · The generator engine (verbatim copy) and a reproduction check
I need the engine for three reasons that all need *new* populations or the hidden state: replicate populations for estimator spread (Sec. 4.5, Sec. 8.5), the ground-truth grading steps (Sec. 4.4, Sec. 4.6, Sec. 6.4, Sec. 9.2) and the sensitivity grid (Sec. 10). The Deliverable 2 notebook cannot be imported, so I copy its engine code here **unchanged** (parameter registry, Phases 1 to 5a, the Phase 5b calibration loop, table assembly, and the estimator helpers). Each block begins with a comment naming the cell it came from. The only addition is that header comment.

**Why trust the copy.** Two tests, both below:
1. With the calibrated parameters from `metadata.json` and seed 193, the copied engine must regenerate the eight tables so that their SHA-256 hashes equal those recorded in `metadata.json`. Because the exported CSV files hash to the same values (Sec. 2), the engine copy and the exported files are the same dataset.
2. Running the verbatim Phase 5b loop on the main population must return the recorded (delta, gamma0).

Hash equality is exact only on the same platform and library versions (WP App. B.3 #5). If it fails on yours, the notebook says so, and the steps that need row-by-row alignment between exports and latent state are skipped.

In [ ]:
# VERBATIM from synthetic-data-generator notebook, cell 4 (parameter registry)
TAGS = {"LOCKED", "CALIBRATED", "ASSUMED"}
PARAM = {}
def reg(name, value, tag, basis):
    assert tag in TAGS, f"{name}: invalid tag {tag}"
    assert name not in PARAM, f"{name}: duplicate"
    PARAM[name] = {"value": value, "tag": tag, "basis": basis}
def P(name): return PARAM[name]["value"]
def set_calibrated(name, value):
    assert PARAM[name]["tag"] == "CALIBRATED", f"{name} is not CALIBRATED; refusing to tune it"
    PARAM[name]["value"] = float(value)

# ---- LOCKED -------------------------------------------------------------------------------------------
reg("T_MAX", 8, "LOCKED", "1 term = 1 semester; 8 terms per 4-year cohort (WP Table 4)")
reg("intervention_effect", 0.30, "LOCKED", "True relative hazard reduction (Eq. 3); internal ground truth, NOT an empirical claim (WP Sec. 9)")
reg("tol_pp", 0.03, "LOCKED", "±3 pp Empirical-Fidelity tolerance (WP Sec. 3). Not tagged explicitly in WP -> treated as design-fixed; see open flag OF-5")
# ---- CALIBRATED (cited figures) -----------------------------------------------------------------------
reg("target_high_2022", 0.671, "CALIBRATED", "Mean of BARMM 93.4, Reg VII 60.7, Reg IX 59.5, CAR 54.9 = 67.1% (CPBRD 2025, Class-of-2022 column; EDCOM II 2025)")
reg("target_typical_2022", 0.459, "CALIBRATED", "Enrollment-weighted mean of the 13 remaining regions, Class of 2022 = 45.9% (CPBRD 2025)")
reg("national_2023_level", 0.294, "CALIBRATED", "CHED national total read as Class of 2023 (CPBRD 2025; CHED FY2026 budget presentation: 29.39%). Held-out level only")
reg("observed_ratio_2023", 1.37, "CALIBRATED", "Class-of-2023 high/typical ratio on the 3 high-tier regions with data (CAR, VII, IX): 35.7/26.0 (WP Sec. 11)")
# ---- CALIBRATED (fitted by the Phase 5b loop; initial = Appendix A pilot) ---------------------------------
reg("delta", 1.06, "CALIBRATED", "Tier shift in baseline vulnerability; fitted by Phase 5b to the high-tier target (WP App. A.1)")
reg("gamma0", -2.67, "CALIBRATED", "Hazard intercept (Eq. 3); fitted by Phase 5b to the typical-tier target (WP App. A.1)")
# ---- ASSUMED: Phase 1 --------------------------------------------------------------------------------
reg("tier_share", 0.15, "ASSUMED", "Share of students in high-attrition tier; within CPBRD-implied 14.1–18% range (WP Table 4)")
reg("work_share", 0.35, "ASSUMED", "Working-student prevalence; directionally supported only (WP Table 4)")
# ---- ASSUMED: Phase 2/3 generative forms (WP Appendix A.1) --------------------------------------------
reg("shock_prob", 0.08, "ASSUMED", "Per-term acute-shock probability (WP Table 4)")
reg("vuln_work", 0.5, "ASSUMED", "v = δ·tier + 0.5·work − 0.4·inc + 0.8·ε (WP A.1)")
reg("vuln_inc", -0.4, "ASSUMED", "Higher income proxy -> lower vulnerability (WP A.1)")
reg("vuln_eps", 0.8, "ASSUMED", "Idiosyncratic vulnerability noise (WP A.1)")
reg("grade_intercept", 72.0, "ASSUMED", "grade = 72 − 7v − 7·shock + 8ε (WP A.1)")
reg("grade_v", -7.0, "ASSUMED", "WP A.1")
reg("grade_shock", -7.0, "ASSUMED", "WP A.1")
reg("grade_noise", 8.0, "ASSUMED", "WP A.1")
reg("att_intercept", 0.88, "ASSUMED", "attendance = 0.88 − 0.06v − 0.08·shock + 0.08ε (WP A.1)")
reg("att_v", -0.06, "ASSUMED", "WP A.1")
reg("att_shock", -0.08, "ASSUMED", "WP A.1")
reg("att_noise", 0.08, "ASSUMED", "WP A.1")
reg("hold_intercept", -2.2, "ASSUMED", "hold ~ Bernoulli(σ(−2.2 + 0.9v + 1.2·shock)) (WP A.1)")
reg("hold_v", 0.9, "ASSUMED", "WP A.1")
reg("hold_shock", 1.2, "ASSUMED", "WP A.1")
# ---- ASSUMED: Phase 4 (Eq. 1–2) ------------------------------------------------------------------------
reg("tau_g", 60.0, "ASSUMED", "Low-grade cutoff on 0–100 scale (WP Table 4)")
reg("tau_a", 0.75, "ASSUMED", "Low-attendance cutoff (WP Table 4)")
reg("beta0", -9.0, "ASSUMED", "Eq. 1 intercept; chosen jointly with beta under illustrative submodel; NOT fitted to pilot population (WP Table 3/4)")
reg("beta", 6.0, "ASSUMED", "Eq. 1 shared weight beta1=beta2=beta3 (equal-weight symmetry assumption, WP Sec. 8)")
reg("design_sensitivity", 0.70, "ASSUMED", "Design target: share of genuinely high-vulnerability student-terms offered support (WP Sec. 8)")
reg("design_false_flag", 0.25, "ASSUMED", "Design target: share of supported student-terms with genuinely low vulnerability (WP Sec. 8)")
# ---- ASSUMED: Phase 5a (Eq. 3) -----------------------------------------------------------------------
reg("gamma1", 0.8, "ASSUMED", "Hazard slope on standardized vulnerability (WP A.1)")
reg("gamma2", 1.5, "ASSUMED", "Hazard slope on shock indicator (WP A.1)")
# ---- ASSUMED: validation thresholds ------------------------------------------------------------------
reg("heldout_refit_tol", 0.01, "ASSUMED", "Held-out refit: pooled attrition within 1 pp of 29.4% (WP Sec. 11)")
reg("heldout_band_lo", 1.2, "ASSUMED", "Held-out tier-ratio band, lower edge (WP Sec. 11)")
reg("heldout_band_hi", 1.7, "ASSUMED", "Held-out tier-ratio band, upper edge (WP Sec. 11)")
reg("recov_abs_window", 0.05, "ASSUMED", "Recoverability: point estimate within max(5 pp, 2·SD across runs) (WP Table 5)")
reg("edge_min_share", 0.01, "ASSUMED", "'Non-trivial' edge-case rate: >= 1% of the relevant event set. WP does not define 'non-trivial' (open flag OF-6)")
reg("edge_min_count", 100, "ASSUMED", "'Non-trivial' edge-case rate: >= 100 student-terms (OF-6)")
reg("targeting_high_current_quantile", 0.70, "ASSUMED", "Diagnostic only: top 30% of hazard linear predictor = 'genuinely high current vulnerability' (matches WP Table 3 illustrative 30% split)")
# ---- ASSUMED: non-core attributes (WP App. B: 'marginal forms unconstrained by this paper') ----------
reg("entry_year", 2018, "ASSUMED", "Single calibrated cohort: Class of 2022 = entered AY 2018–19. Constant by design (OF-3)")
reg("program_shares", {"Business & Management": 0.22, "Education": 0.14, "Engineering & IT": 0.24,
                       "Health Sciences": 0.12, "Arts, Sciences & Social Sci.": 0.20, "Other": 0.08},
    "ASSUMED", "Programme mix; causally inert in this version (affects course load only)")
reg("program_courses", {"Business & Management": 6.0, "Education": 6.0, "Engineering & IT": 6.5,
                        "Health Sciences": 7.0, "Arts, Sciences & Social Sci.": 6.0, "Other": 6.0},
    "ASSUMED", "Mean courses per term by programme")
reg("course_noise_sd", 0.8, "ASSUMED", "SD of per-term course-count noise (counts are clipped to courses_min..courses_max)")
reg("pass_midpoint", 62.0, "ASSUMED", "P(pass course) = σ((grade − 62)/4)")
reg("pass_scale", 4.0, "ASSUMED", "see above")
reg("dist_median_km", 8.0, "ASSUMED", "distance_km = 8·exp(0.35·tier + 0.10·v + 0.8z)")
reg("dist_tier", 0.35, "ASSUMED", "High-tier students live further out on average")
reg("dist_v", 0.10, "ASSUMED", "Weak link between distance and latent vulnerability (permitted by App. B.2 leakage rule)")
reg("dist_sigma", 0.8, "ASSUMED", "Log-scale spread of distance")
reg("subsidy_by_bracket", {1: 0.45, 2: 0.35, 3: 0.20, 4: 0.10, 5: 0.05}, "ASSUMED", "P(subsidy at entry | income bracket 1=lowest)")
reg("subsidy_lapse_prob", 0.25, "ASSUMED", "Per-term P(permanent subsidy lapse) when the term grade < tau_g")
reg("overdue_median_php", 5000.0, "ASSUMED", "Median overdue amount (PHP) when a hold is active")
reg("overdue_v", 0.15, "ASSUMED", "Overdue amount rises mildly with vulnerability")
reg("overdue_sigma", 0.6, "ASSUMED", "Log-scale spread of overdue amount")
reg("n_income_brackets", 5, "ASSUMED", "Number of income brackets (quintiles of the standardized income proxy; 1 = lowest income)")
reg("units_per_course", 3, "ASSUMED", "Units per course: units_attempted = units_per_course x courses taken")
reg("courses_min", 3, "ASSUMED", "Minimum courses per term (lower clip bound for the per-term course count)")
reg("courses_max", 9, "ASSUMED", "Maximum courses per term (upper clip bound; also the width of the course-level uniform draws)")

assert P("beta") > 0 and P("gamma1") > 0 and P("gamma2") > 0          # sign constraints in Eq. 1 and 3
assert set(P("subsidy_by_bracket")) == set(range(1, P("n_income_brackets") + 1))   # one subsidy probability per income bracket
param_log = pd.DataFrame([{"parameter": k, "value": json.dumps(v["value"]) if isinstance(v["value"], dict) else v["value"],
                           "tag": v["tag"], "basis": v["basis"]} for k, v in PARAM.items()])
assert param_log["tag"].isin(TAGS).all()
print(param_log["tag"].value_counts().to_string())
display(param_log[param_log.tag != "ASSUMED"][["parameter", "value", "tag", "basis"]].style.hide(axis="index"))

In [ ]:
# VERBATIM from synthetic-data-generator notebook, cell 6 (engine, Phases 1-5a)
sig = lambda x: 1.0 / (1.0 + np.exp(-x))

STREAMS = ["tier","work","inc","eps","shock","eg","ea","hold","assign","drop",
           "program","dist","subsidy","lapse","units","courses","overdue"]
MAX_COURSES = int(P("courses_max"))                      # width of the course-level uniform draws

def draw_noise(N, T, entropy, noncore=True):
    # All random draws, once, from independent named streams (common random numbers).
    ss = np.random.SeedSequence(entropy)
    rng = {n: np.random.default_rng(c) for n, c in zip(STREAMS, ss.spawn(len(STREAMS)))}
    d = dict(
        tier   = rng["tier"].random(N) < P("tier_share"),            # Phase 1
        work   = rng["work"].random(N) < P("work_share"),
        inc    = rng["inc"].standard_normal(N),
        eps    = rng["eps"].standard_normal(N),
        shock  = rng["shock"].random((N, T)) < P("shock_prob"),      # Phase 2 (term shock, independent of everything)
        eg     = rng["eg"].standard_normal((N, T)),                  # Phase 3 noise
        ea     = rng["ea"].standard_normal((N, T)),
        u_hold = rng["hold"].random((N, T)),
        u_assign = rng["assign"].random((N, T)),                     # Phase 4
        u_drop = rng["drop"].random((N, T)),                         # Phase 5a
    )
    if noncore:
        d.update(program_u = rng["program"].random(N), z_dist = rng["dist"].standard_normal(N),
                 u_subsidy = rng["subsidy"].random(N), u_lapse = rng["lapse"].random((N, T)),
                 z_units = rng["units"].standard_normal((N, T)),
                 u_course = rng["courses"].random((N, T, MAX_COURSES)),
                 z_overdue = rng["overdue"].standard_normal((N, T)))
    return d

def simulate_latent(d, delta):
    # Phases 2–4: baseline vulnerability, observables, risk score (Eq.1), intervention (Eq.2).
    N, T = d["shock"].shape
    v_raw = (delta * d["tier"] + P("vuln_work") * d["work"] + P("vuln_inc") * d["inc"] + P("vuln_eps") * d["eps"])
    v = (v_raw - v_raw.mean()) / v_raw.std()                          # standardized across the cohort
    vc, sh = v[:, None], d["shock"].astype(float)
    grade = P("grade_intercept") + P("grade_v")*vc + P("grade_shock")*sh + P("grade_noise")*d["eg"]
    att   = P("att_intercept")  + P("att_v")*vc    + P("att_shock")*sh   + P("att_noise")*d["ea"]
    hold  = d["u_hold"] < sig(P("hold_intercept") + P("hold_v")*vc + P("hold_shock")*sh)
    low_g, low_a = grade < P("tau_g"), att < P("tau_a")               # comparisons on UNclipped values
    b1 = b2 = b3 = P("beta")
    risk = np.full((N, T), np.nan)                                    # undefined in term 1
    risk[:, 1:] = sig(P("beta0") + b1*low_g[:, :-1] + b2*low_a[:, :-1] + b3*hold[:, :-1])
    interv = np.zeros((N, T), bool)                                   # no assignment in term 1
    interv[:, 1:] = d["u_assign"][:, 1:] < risk[:, 1:]
    cell = np.zeros((N, T), int)                                      # prior-term signal cell (4*lowG + 2*lowA + hold)
    cell[:, 1:] = 4*low_g[:, :-1] + 2*low_a[:, :-1] + hold[:, :-1]
    return dict(v=v, shock=d["shock"], grade=grade, att=att, hold=hold, low_g=low_g, low_a=low_a,
                risk=risk, interv=interv, cell=cell)

def apply_hazard(L, u_drop, gamma0):
    # Phase 5a: Eq. 3–4. Returns potential-dropout matrix, attrition flag, dropout/censor term, hazard.
    N, T = u_drop.shape
    lin = gamma0 + P("gamma1") * L["v"][:, None] + P("gamma2") * L["shock"]
    hz = sig(lin) * (1.0 - P("intervention_effect") * L["interv"])
    dp = u_drop < hz
    attr = dp.any(axis=1)
    first = np.where(attr, dp.argmax(axis=1) + 1, T)                  # T_i: dropout term, or last term if censored
    return dp, attr, first, hz

def bisect(f, lo, hi, target, n=45):
    # f must be non-decreasing; returns x with f(x) ~ target
    assert f(lo) <= target <= f(hi), "target not bracketed; widen the search interval"
    for _ in range(n):
        mid = 0.5 * (lo + hi)
        if f(mid) < target: lo = mid
        else: hi = mid
    return 0.5 * (lo + hi)
print("engine defined")

In [ ]:
# VERBATIM from synthetic-data-generator notebook, cell 8 (Phase 5b calibration function only)
def calibrate(d, verbose=True):
    tier = d["tier"]
    def inner(delta):
        L = simulate_latent(d, delta)
        g0 = bisect(lambda g: apply_hazard(L, d["u_drop"], g)[1][~tier].mean(), -10, 5, P("target_typical_2022"))
        high = apply_hazard(L, d["u_drop"], g0)[1][tier].mean()
        return g0, high
    delta = bisect(lambda dl: inner(dl)[1], 0.0, 4.0, P("target_high_2022"), n=28)
    g0, high = inner(delta)
    return delta, g0

In [ ]:
# VERBATIM from synthetic-data-generator notebook, cell 10 (table assembly) and cell 11 (hash helper)
def build_tables(d, L, gamma0):
    N, T = L["grade"].shape
    dp, attr, first, hz = apply_hazard(L, d["u_drop"], gamma0)
    tix = np.arange(1, T + 1)
    mask = tix[None, :] <= first[:, None]                 # rows exist for terms 1..T_i only
    r, c = np.nonzero(mask)                               # row-major => ordered by student then term
    sid, term = r + 1, c + 1
    tier = d["tier"]

    # ---- Students (fixed) -------------------------------------------------------------------------
    income_bracket = pd.qcut(d["inc"], P("n_income_brackets"), labels=False).astype(int) + 1     # 1 = lowest income
    names = list(P("program_shares")); cum = np.cumsum(list(P("program_shares").values()))
    prog = np.array(names)[np.minimum(np.searchsorted(cum, d["program_u"], side="right"), len(names) - 1)]
    distance = np.exp(np.log(P("dist_median_km")) + P("dist_tier")*tier + P("dist_v")*L["v"] + P("dist_sigma")*d["z_dist"])
    p_sub = np.array([P("subsidy_by_bracket")[b] for b in range(1, P("n_income_brackets") + 1)])[income_bracket - 1]
    sub0 = d["u_subsidy"] < p_sub
    Students = pd.DataFrame({"student_id": np.arange(1, N + 1),
        "region_tier": np.where(tier, "high", "typical"), "entry_year": P("entry_year"), "program": prog,
        "income_bracket": income_bracket, "working_student": d["work"],
        "distance_km": distance.round(2), "subsidy_at_entry": sub0})
    Regions = pd.DataFrame({"region_tier": ["high", "typical"], "tier_share": [P("tier_share"), 1 - P("tier_share")],
        "target_attrition": [P("target_high_2022"), P("target_typical_2022")],
        "target_source": ["CPBRD 2025 (CHED FY2026 Agency Budget Notes), Class-of-2022 column: mean of BARMM, VII, IX, CAR; EDCOM II 2025",
                          "CPBRD 2025, Class-of-2022 column: enrollment-weighted mean of the 13 remaining regions"]})

    # ---- panel observables --------------------------------------------------------------------------
    grade = np.clip(L["grade"][mask], 0, 100); att = np.clip(L["att"][mask], 0, 1); hold = L["hold"][mask]
    course_mean = pd.Series(prog).map(P("program_courses")).to_numpy()[r]
    n_course = np.clip(np.rint(course_mean + P("course_noise_sd") * d["z_units"][mask]), P("courses_min"), P("courses_max")).astype(int)
    p_pass = sig((grade - P("pass_midpoint")) / P("pass_scale"))
    passed = ((d["u_course"][mask] < p_pass[:, None]) & (np.arange(MAX_COURSES)[None, :] < n_course[:, None])).sum(1)
    lapse_ev = (L["grade"] < P("tau_g")) & (d["u_lapse"] < P("subsidy_lapse_prob"))
    lapsed = np.maximum.accumulate(lapse_ev, axis=1)                           # lapse is permanent
    sub_status = (sub0[:, None] & ~lapsed)[mask]
    overdue = np.where(hold, np.exp(np.log(P("overdue_median_php")) + P("overdue_v")*L["v"][r]
                                    + P("overdue_sigma")*d["z_overdue"][mask]), 0.0)
    drop_flag = (term == first[r]) & attr[r]

    key = {"student_id": sid, "term": term}
    Enrollments = pd.DataFrame({**key, "enrollment_status": np.where(drop_flag, "withdrawn", "continuing"),
                                "censored": (term == T) & ~attr[r]})
    CourseGrades = pd.DataFrame({**key, "term_avg_score": grade.round(4), "units_attempted": P("units_per_course") * n_course,
                                 "units_passed": P("units_per_course") * passed})
    AttendanceRecords = pd.DataFrame({**key, "attendance_rate": att.round(6)})
    FinancialHolds = pd.DataFrame({**key, "hold_flag": hold, "overdue_amount": overdue.round(2), "subsidy_status": sub_status})
    SupportInterventions = pd.DataFrame({**key, "risk_score": L["risk"][mask], "intervention": L["interv"][mask]})
    PersistenceOutcomes = pd.DataFrame({**key, "dropout_flag": drop_flag, "attrition_flag": attr[r]})

    tables = {"Regions": Regions, "Students": Students, "Enrollments": Enrollments, "CourseGrades": CourseGrades,
              "AttendanceRecords": AttendanceRecords, "FinancialHolds": FinancialHolds,
              "SupportInterventions": SupportInterventions, "PersistenceOutcomes": PersistenceOutcomes}
    audit = {"student": pd.DataFrame({"student_id": np.arange(1, N + 1), "baseline_vulnerability": L["v"]}),
             "panel": pd.DataFrame({**key, "shock": L["shock"][mask], "hazard": hz[mask]})}
    return tables, audit

def sha256_df(df): return hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()

In [ ]:
# VERBATIM from synthetic-data-generator notebook, cell 17 (analyst-view frame and estimator helpers)
def analyst_frame(tb):
    df = (tb["SupportInterventions"]
          .merge(tb["CourseGrades"][["student_id","term","term_avg_score"]], on=["student_id","term"])
          .merge(tb["AttendanceRecords"][["student_id","term","attendance_rate"]], on=["student_id","term"])
          .merge(tb["FinancialHolds"][["student_id","term","hold_flag"]], on=["student_id","term"])
          .merge(tb["PersistenceOutcomes"][["student_id","term","dropout_flag"]], on=["student_id","term"])
          .sort_values(["student_id","term"]).reset_index(drop=True))
    g = df.groupby("student_id")
    lg, la, lh = g["term_avg_score"].shift(1), g["attendance_rate"].shift(1), g["hold_flag"].shift(1)
    df["cell"] = (4*(lg < P("tau_g")) + 2*(la < P("tau_a")) + lh.astype(float)).where(df.term >= 2)
    return df[df.term >= 2].assign(cell=lambda x: x.cell.astype(int))

def mh_rr(y, Tr, cell):
    # Mantel–Haenszel risk ratio (treated / control) and Greenland–Robins 95% CI on log RR
    K = 8
    a  = np.bincount(cell, weights=(y & Tr),  minlength=K); n1 = np.bincount(cell, weights=Tr.astype(float),  minlength=K)
    c_ = np.bincount(cell, weights=(y & ~Tr), minlength=K); n0 = np.bincount(cell, weights=(~Tr).astype(float), minlength=K)
    n = n1 + n0; ok = n > 0
    R = (a[ok]*n0[ok]/n[ok]).sum(); S_ = (c_[ok]*n1[ok]/n[ok]).sum(); rr = R / S_
    var = ((n1[ok]*n0[ok]*(a[ok]+c_[ok]) - a[ok]*c_[ok]*n[ok]) / n[ok]**2).sum() / (R * S_)
    se = np.sqrt(var); return rr, (rr*np.exp(-1.96*se), rr*np.exp(1.96*se))

def poisson_rr(y, Tr, cell, groups):
    X = pd.get_dummies(pd.Series(cell), prefix="c", drop_first=True, dtype=float)
    X.insert(0, "T", Tr.astype(float)); X = sm.add_constant(X, has_constant="add")
    g = pd.factorize(groups)[0]
    m = sm.GLM(y.astype(float), X, family=sm.families.Poisson()).fit(cov_type="cluster", cov_kwds={"groups": g})
    b, se = m.params["T"], m.bse["T"]; return np.exp(b), (np.exp(b - 1.96*se), np.exp(b + 1.96*se))

def logit_or(y, Tr, cell):
    X = pd.get_dummies(pd.Series(cell), prefix="c", drop_first=True, dtype=float)
    X.insert(0, "T", Tr.astype(float)); X = sm.add_constant(X, has_constant="add")
    return float(np.exp(sm.Logit(y.astype(float), X).fit(disp=0).params["T"]))

def ipw_rr(y, Tr, p, clip=None, hajek=True):
    # IPW risk ratio with the TRUE propensity (exported as risk_score). hajek=True normalises weights; False = Horvitz–Thompson.
    p = np.clip(p, *clip) if clip else p
    w = np.where(Tr, 1/p, 1/(1-p))
    if hajek: m1 = (w*Tr*y).sum() / (w*Tr).sum();  m0 = (w*(~Tr)*y).sum() / (w*(~Tr)).sum()
    else:     m1 = (Tr*y/p).mean();                m0 = ((~Tr)*y/(1-p)).mean()
    return m1 / m0, w.max()

def boot_naive(y, Tr, sid, B, seed):
    u, inv = np.unique(sid, return_inverse=True); M = len(u)
    E1 = np.bincount(inv, weights=(y & Tr), minlength=M);  N1 = np.bincount(inv, weights=Tr.astype(float), minlength=M)
    E0 = np.bincount(inv, weights=(y & ~Tr), minlength=M); N0 = np.bincount(inv, weights=(~Tr).astype(float), minlength=M)
    rng = np.random.default_rng(seed); out = np.empty(B)
    for b in range(B):
        s = rng.integers(0, M, M); out[b] = 1 - (E1[s].sum()/N1[s].sum()) / (E0[s].sum()/N0[s].sum())
    return np.percentile(out, [2.5, 97.5])

In [ ]:
# ---- reproduction test 1: regenerate the dataset from the recorded calibration and compare hashes ----------
assert (N, T) == (META["n_students"], P("T_MAX"))
set_calibrated("delta", META["calibrated"]["delta"]); set_calibrated("gamma0", META["calibrated"]["gamma0"])
TIX = np.arange(1, T + 1)
t0 = time.time()
d0 = draw_noise(N, T, [SEED, 0])
L0 = simulate_latent(d0, P("delta"))
TB_ENG, AUD = build_tables(d0, L0, P("gamma0"))
PARITY_EXACT = all(sha256_df(TB_ENG[n]) == META["sha256"][n] for n in TABLE_NAMES)
par = pd.DataFrame([{"table": n, "regenerated rows": len(TB_ENG[n]), "SHA-256 equals metadata": sha256_df(TB_ENG[n]) == META["sha256"][n]} for n in TABLE_NAMES])
display(par.style.hide(axis="index"))
print(f"Regenerated in {time.time()-t0:.0f}s | exact reproduction of all eight tables: {PARITY_EXACT}")
GT_OK = PARITY_EXACT and bool(integ["file SHA-256 matches metadata"].all())
if not GT_OK:
    print("WARNING: exact reproduction is not confirmed on this platform. Ground-truth steps that need row alignment (Sec. 6.4, Sec. 9.2) will be skipped; "
          "the latent-state checks that do not need alignment still run on the regenerated tables.")

# ---- reproduction test 2: the verbatim calibration loop must return the recorded (delta, gamma0) ------------
if not FAST:
    t0 = time.time(); dl, g0_ = calibrate(d0)
    print(f"Phase 5b re-run in {time.time()-t0:.0f}s: delta = {dl:.12f} (recorded {META['calibrated']['delta']:.12f}); gamma0 = {g0_:.12f} (recorded {META['calibrated']['gamma0']:.12f})")
    assert abs(dl - META["calibrated"]["delta"]) < 1e-9 and abs(g0_ - META["calibrated"]["gamma0"]) < 1e-9
    print("✔ the copied calibration loop reproduces the recorded calibration")
else:
    print("FAST_MODE: calibration re-run skipped")

## 4 · Is the generator valid? Re-derive the checks from the CSVs
Deliverable 2 reported a validation scorecard. Here I do not take it on trust: I re-derive each check that can be computed from the exported tables (plus, where the check is about the hidden state, from the regenerated latent state), and at the end I reconcile my numbers against the scorecard file the generator wrote (Sec. 4.7). Three checks are not re-run here: the held-out Class-of-2023 check, parameter traceability, and the generator's own sensitivity scan. They belong to the generator notebook (Sec. 7 there), and I carry the held-out fragility forward in Sec. 12.

**Calibration is not validation** (WP Sec. 11). Matching the two tier targets only shows that the Phase 5b loop converged. The checks with independent content are the correlation structure, the confounding check and the recoverability check.

### 4.1 Tier attrition against the calibration targets
Class-of-2022 targets (`CALIBRATED`): high tier 67.1 %, typical tier 45.9 %; tolerance ±3 pp (`LOCKED`). The pooled rate has no cited target; it is the blend implied by the tier share.

In [ ]:
S = TB["Students"].set_index("student_id")
PO = TB["PersistenceOutcomes"]
attr_s = PO.groupby("student_id").attrition_flag.first()
rate_hi, rate_ty, rate_all = (attr_s[S.region_tier == "high"].mean(), attr_s[S.region_tier == "typical"].mean(), attr_s.mean())
tol = P("tol_pp"); blend = P("tier_share") * P("target_high_2022") + (1 - P("tier_share")) * P("target_typical_2022")
cal = pd.DataFrame([
    {"measure": "Tier attrition: high",    "measured": rate_hi, "target": P("target_high_2022"),    "gap (pp)": 100*(rate_hi - P("target_high_2022"))},
    {"measure": "Tier attrition: typical", "measured": rate_ty, "target": P("target_typical_2022"), "gap (pp)": 100*(rate_ty - P("target_typical_2022"))},
    {"measure": "National attrition (derived blend)", "measured": rate_all, "target": blend, "gap (pp)": 100*(rate_all - blend)}])
cal["within ±3 pp"] = cal["gap (pp)"].abs() <= 100 * tol
banner(); display(cal.style.format({"measured": "{:.1%}", "target": "{:.1%}", "gap (pp)": "{:+.2f}"}).hide(axis="index"))

### 4.2 Marginals and ranges
Do the realised marginals match the design values, within sampling error, and are the observed ranges sensible? A discrepancy larger than about three standard errors would mean the export and the registry disagree.

In [ ]:
n_ = len(S)
def se_p(p): return np.sqrt(p * (1 - p) / n_)
sub_tbl = S.groupby("income_bracket").subsidy_at_entry.mean()
marg = pd.DataFrame([
    {"quantity": "High-tier share", "observed": S.region_tier.eq("high").mean(), "design value": P("tier_share"), "SE": se_p(P("tier_share"))},
    {"quantity": "Working-student share", "observed": S.working_student.mean(), "design value": P("work_share"), "SE": se_p(P("work_share"))},
    {"quantity": "Income bracket 1 share (should be 1/5)", "observed": (S.income_bracket == 1).mean(), "design value": 0.2, "SE": se_p(0.2)},
    {"quantity": "Subsidy at entry, bracket 1", "observed": sub_tbl.loc[1], "design value": P("subsidy_by_bracket")[1], "SE": np.sqrt(P("subsidy_by_bracket")[1]*(1-P("subsidy_by_bracket")[1]) / (n_/5))},
    {"quantity": "Subsidy at entry, bracket 5", "observed": sub_tbl.loc[5], "design value": P("subsidy_by_bracket")[5], "SE": np.sqrt(P("subsidy_by_bracket")[5]*(1-P("subsidy_by_bracket")[5]) / (n_/5))},
])
marg["z"] = (marg.observed - marg["design value"]) / marg.SE; marg["within 3 SE"] = marg.z.abs() < 3
banner(); display(marg.style.format({"observed": "{:.4f}", "design value": "{:.4f}", "SE": "{:.4f}", "z": "{:+.2f}"}).hide(axis="index"))

cg, ar, fh, si = TB["CourseGrades"], TB["AttendanceRecords"], TB["FinancialHolds"], TB["SupportInterventions"]
rng_tbl = pd.DataFrame({
    "term_avg_score": cg.term_avg_score.describe()[["min", "mean", "max"]],
    "attendance_rate": ar.attendance_rate.describe()[["min", "mean", "max"]],
    "overdue_amount (PHP, hold terms only)": fh.overdue_amount[fh.hold_flag].describe()[["min", "mean", "max"]],
    "distance_km": S.distance_km.describe()[["min", "mean", "max"]]}).T
print("Ranges:"); display(rng_tbl.style.format("{:,.2f}"))
print(f"Hold prevalence: {fh.hold_flag.mean():.1%} of student-terms | support rate (t>=2): {si[si.term >= 2].intervention.mean():.1%} | "
      f"mean terms observed per student: {len(TB['Enrollments'])/N:.2f}")

### 4.3 Correlation structure: the observed signals
The whitepaper (Sec. 7) requires grades, attendance and holds to be **correlated unconditionally** (an analyst sees one cloud of "struggling" students) but **independent given the latent state**. Sec. 4.3 computes the first half from the exports. Sec. 4.4 computes the second half and is one of the places where I use the hidden state to grade the data.

In [ ]:
pan = (cg[["student_id", "term", "term_avg_score"]].merge(ar[["student_id", "term", "attendance_rate"]], on=["student_id", "term"])
       .merge(fh[["student_id", "term", "hold_flag"]], on=["student_id", "term"]))
cols = {"grade": pan.term_avg_score.to_numpy(float), "attendance": pan.attendance_rate.to_numpy(float), "hold": pan.hold_flag.to_numpy(float)}
unc = pd.DataFrame(cols).corr()
banner(); print("Unconditional correlations of the three signals (what an analyst sees; student-term level):"); display(unc.round(3))

### 4.4 Correlation structure given the latent state *(uses the hidden state to grade the data)*
I remove the linear effect of baseline vulnerability and the current shock from each signal, using the regenerated latent state (`AUD`), and re-correlate the residuals. If the generator is faithful to Figure 2 of the whitepaper, the unconditional correlations survive in Sec. 4.3 and these residual correlations are near zero.

In [ ]:
dp0, at0, fi0, hz0 = apply_hazard(L0, d0["u_drop"], P("gamma0"))
pan_e = (TB_ENG["CourseGrades"][["student_id", "term", "term_avg_score"]].merge(TB_ENG["AttendanceRecords"][["student_id", "term", "attendance_rate"]], on=["student_id", "term"])
         .merge(TB_ENG["FinancialHolds"][["student_id", "term", "hold_flag"]], on=["student_id", "term"])
         .merge(AUD["panel"][["student_id", "term", "shock"]], on=["student_id", "term"]).merge(AUD["student"], on="student_id"))
def resid(y_, X_):
    X_ = sm.add_constant(X_.astype(float)); return y_ - X_ @ np.linalg.lstsq(X_, y_, rcond=None)[0]
Xl = pan_e[["baseline_vulnerability", "shock"]].to_numpy()
cols_e = {"grade": pan_e.term_avg_score.to_numpy(float), "attendance": pan_e.attendance_rate.to_numpy(float), "hold": pan_e.hold_flag.to_numpy(float)}
con = pd.DataFrame({k: resid(v, Xl) for k, v in cols_e.items()}).corr()
off = lambda m: m.values[np.triu_indices(3, 1)]
print("Correlations after removing the latent state (hidden from analysts):"); display(con.round(3))
corr_ok = np.abs(off(unc)).min() > 0.10 and np.abs(off(con)).max() < 0.05
print(f"Smallest |unconditional| = {np.abs(off(unc)).min():.3f} (needs > 0.10); largest |conditional| = {np.abs(off(con)).max():.3f} (needs < 0.05) → {'PASS' if corr_ok else 'FAIL'}")

### 4.5 Positivity, estimator recovery and replicate populations
This block is the core of the generator's validation. The analyst frame is rebuilt from the exports (prior-term signals reconstructed from the three signal tables) and checked against the engine's internal arrays; then the estimators are run on the exported data, and on **20 independent populations** (`LOCKED` by WP Table 5) to measure how much each estimator moves from one population to another.

**Estimand.** The true effect is a *relative hazard reduction of 30 %*, which is a **risk ratio** of 0.70 and *not* an odds ratio (WP Sec. 11). Every estimate below is reported as the relative reduction, 1 minus the ratio. A negative value means supported students dropped out *more* often.

In [ ]:
AF = analyst_frame(TB)                                           # verbatim helper: t >= 2 rows, prior-term signal cell
y = AF.dropout_flag.to_numpy(bool); Tr = AF.intervention.to_numpy(bool); cell = AF.cell.to_numpy(int)
sid = AF.student_id.to_numpy(); prop = AF.risk_score.to_numpy(float)
kk = ((cell >> 2) & 1) + ((cell >> 1) & 1) + (cell & 1)          # number of prior-term signals that fired (0..3)

def arrays_from_engine(L, dp, first):
    m_all = TIX[None, :] <= first[:, None]; m2 = m_all & (TIX[None, :] >= 2)
    return (dp & m_all)[m2], L["interv"][m2], L["cell"][m2], L["risk"][m2], np.nonzero(m2)[0], m2

y0_, Tr0_, c0_, p0_, s0_, m2_0 = arrays_from_engine(L0, dp0, fi0)
assert len(AF) == len(y0_) and (cell == c0_).all() and (Tr == Tr0_).all() and (y == y0_).all() and np.allclose(prop, p0_)
print(f"✔ the analyst view rebuilt from the CSV exports equals the engine's internal arrays on {len(AF):,} student-terms with t ≥ 2")

pos = (AF.assign(k=kk).groupby("k").agg(student_terms=("intervention", "size"), supported=("intervention", "sum"))
         .assign(not_supported=lambda x: x.student_terms - x.supported, share_supported=lambda x: x.supported / x.student_terms))
banner(); print("Positivity: how many student-terms of each prior-signal count were supported or not"); display(pos.style.format({"student_terms": "{:,}", "supported": "{:,}", "not_supported": "{:,}", "share_supported": "{:.2%}"}))
BOTH = [int(k) for k in pos.index[(pos.supported >= Q("min_arm")) & (pos.not_supported >= Q("min_arm"))]]
print(f"Signal counts where both arms have ≥ {Q('min_arm')} student-terms: {BOTH}")

The shape of the table is the central fact of this dataset. With the default assignment rule, a student-term with no signal is essentially never supported (26 of 186,653), one with three signals is always supported (2,237 of 2,237), and only the one-signal and two-signal groups contain both arms. Everything that depends on comparing supported with unsupported students therefore rests on those two groups. Sec. 8 and Sec. 10 return to this.

In [ ]:
rel = lambda rr: 1 - rr                                                      # relative reduction = 1 - ratio
ESTLAB = {"naive": "Naive (unadjusted)", "mh": "Mantel–Haenszel RR (8 cells)", "poisson": "Poisson, cluster-robust (8 cells)",
          "ipw_hajek": "IPW Hajek, untrimmed", "ipw_hajek_clip": "IPW Hajek, clipped [.05,.95]",
          "ipw_ht": "IPW Horvitz–Thompson, untrimmed", "ipw_ht_clip": "IPW Horvitz–Thompson, clipped [.05,.95]", "logit_or": "Logistic odds ratio (cell-adjusted)"}

def estimator_suite(y, Tr, cell, p, sid, poisson=True):
    out = {"naive": rel(y[Tr].mean() / y[~Tr].mean()), "mh": rel(mh_rr(y, Tr, cell)[0])}
    if poisson: out["poisson"] = rel(poisson_rr(y, Tr, cell, sid)[0])
    out["ipw_hajek"] = rel(ipw_rr(y, Tr, p)[0]);                      out["ipw_hajek_clip"] = rel(ipw_rr(y, Tr, p, (0.05, 0.95))[0])
    out["ipw_ht"] = rel(ipw_rr(y, Tr, p, hajek=False)[0]);           out["ipw_ht_clip"] = rel(ipw_rr(y, Tr, p, (0.05, 0.95), hajek=False)[0])
    out["logit_or"] = rel(logit_or(y, Tr, cell))
    return out

TRUE_EFFECT = P("intervention_effect")
EM = estimator_suite(y, Tr, cell, prop, sid)                      # main population, from the exported CSVs
naive_ci = boot_naive(y, Tr, sid, Q("n_bootstrap"), SEED + 7)
mh_pt, mh_ci = mh_rr(y, Tr, cell); po_pt, po_ci = poisson_rr(y, Tr, cell, sid)
CI = {"naive": tuple(naive_ci), "mh": (rel(mh_ci[1]), rel(mh_ci[0])), "poisson": (rel(po_ci[1]), rel(po_ci[0]))}
print(f"True effect (LOCKED, by construction): {TRUE_EFFECT:.0%} relative reduction")
for k_, v_ in EM.items(): print(f"  {ESTLAB[k_]:<42s} {v_:+7.1%}" + (f"   95% CI {CI[k_][0]:+.1%} to {CI[k_][1]:+.1%}" if k_ in CI else ""))

In [ ]:
t0 = time.time(); reps = []
for r_ in range(1, Q("n_replicates") + 1):
    d_r = draw_noise(N, T, [SEED, r_], noncore=False)
    L_r = simulate_latent(d_r, P("delta")); dp_r, at_r, fi_r, _ = apply_hazard(L_r, d_r["u_drop"], P("gamma0"))
    y_r, T_r, c_r, p_r, s_r, _ = arrays_from_engine(L_r, dp_r, fi_r)
    reps.append({"rep": r_, "high": at_r[d_r["tier"]].mean(), "typical": at_r[~d_r["tier"]].mean(), "pooled": at_r.mean(),
                 **estimator_suite(y_r, T_r, c_r, p_r, s_r)})
REP = pd.DataFrame(reps)
print(f"{len(REP)} independent populations (seeds [{SEED}, 1..{len(REP)}]) in {time.time()-t0:.0f}s")
tab = REP[list(ESTLAB)].agg(["mean", "std", "min", "max"]).T.rename(index=ESTLAB)
banner(); display(tab.style.format("{:+.1%}"))

In [ ]:
# ---- verdicts re-derived from the exports (same definitions as the generator's Table 5 checks) ---------------
sd_runs = REP["mh"].std(ddof=1); window = max(P("recov_abs_window"), 2 * sd_runs)
conf_ok = not (CI["naive"][0] <= TRUE_EFFECT <= CI["naive"][1])
V = [("National attrition (derived blend)", f"{rate_all:.1%}", f"{blend:.1%} ± {tol:.0%}", abs(rate_all - blend) <= tol),
     ("Tier attrition: high", f"{rate_hi:.1%}", f"{P('target_high_2022'):.1%} ± {tol:.0%}", abs(rate_hi - P("target_high_2022")) <= tol),
     ("Tier attrition: typical", f"{rate_ty:.1%}", f"{P('target_typical_2022'):.1%} ± {tol:.0%}", abs(rate_ty - P("target_typical_2022")) <= tol),
     ("Confounding presence", f"naive {EM['naive']:.1%} (95% CI {CI['naive'][0]:.1%} to {CI['naive'][1]:.1%})", f"CI excludes true {TRUE_EFFECT:.0%}", conf_ok)]
for nm, key in [("Mantel–Haenszel", "mh"), ("Poisson (cluster-robust)", "poisson")]:
    pt, ci = EM[key], CI[key]; ok = (ci[0] <= TRUE_EFFECT <= ci[1]) and abs(pt - TRUE_EFFECT) <= window
    V.append((f"Recoverability: {nm}", f"{pt:.1%} (95% CI {ci[0]:.1%} to {ci[1]:.1%})", f"CI ∋ {TRUE_EFFECT:.0%}; |err| ≤ {window:.1%} (=max(5pp, 2×SD {sd_runs:.1%}))", ok))
V.append(("Correlation structure (Sec. 7 of WP)", f"min |unc.| {np.abs(off(unc)).min():.2f}; max |cond.| {np.abs(off(con)).max():.3f}", "unc. > 0.10; cond. < 0.05", corr_ok))
VER = pd.DataFrame(V, columns=["measure", "measured", "benchmark", "pass"]); VER["verdict"] = np.where(VER["pass"], "PASS", "FAIL")
banner(); display(VER[["measure", "measured", "benchmark", "verdict"]].style.hide(axis="index"))
if window > 0.10: print(f"CAUTION: the recoverability window is ±{window:.0%}; at this size the check has limited power.")

### 4.6 Deliberate edge cases *(uses the hidden state to grade the data)*
WP Sec. 12 requires two edge cases to arise from the generative mechanism, not from inserted rows: (1) **an unflagged high-risk student**, meaning a clean prior-term record, no support, and a shock-driven dropout; (2) **a falsely flagged student**, meaning support triggered by a single weak prior-term signal despite below-average baseline vulnerability. "Non-trivial" is defined in the generator (≥ 1 % of the relevant event set and ≥ 100 student-terms, `ASSUMED`).

In [ ]:
v_st = np.repeat(L0["v"][:, None], T, axis=1)[m2_0]; sh_st = L0["shock"][m2_0]
e1 = (c0_ == 0) & ~Tr0_ & sh_st & y0_                           # clean record, no support, shock, dropout
kc0 = ((c0_ >> 2) & 1) + ((c0_ >> 1) & 1) + (c0_ & 1)
e2 = Tr0_ & (kc0 == 1) & (v_st < 0)                             # supported on one weak signal, below-average vulnerability
n_drop0, n_sup0 = int(y0_.sum()), int(Tr0_.sum())
edge = pd.DataFrame([
    {"edge case": "1. Unflagged high-risk (shock-driven dropout, clean record, no support)", "student-terms": int(e1.sum()), "distinct students": len(np.unique(s0_[e1])), "share of": "all dropout events", "share": e1.sum() / n_drop0},
    {"edge case": "2. Falsely flagged (one weak signal, v < 0, supported)", "student-terms": int(e2.sum()), "distinct students": len(np.unique(s0_[e2])), "share of": "all supported student-terms", "share": e2.sum() / n_sup0}])
edge_ok = bool(e1.sum() >= P("edge_min_count") and e1.sum() / n_drop0 >= P("edge_min_share") and e2.sum() >= P("edge_min_count") and e2.sum() / n_sup0 >= P("edge_min_share"))
banner(); display(edge.style.format({"student-terms": "{:,}", "distinct students": "{:,}", "share": "{:.1%}"}).hide(axis="index"))
print("Edge case 1 is only partly visible in the exports: the shock also shaped that term's own grade, attendance and hold (WP Sec. 12).")

### 4.7 Reconciliation with the generator's scorecard
`validation_scorecard.csv` was written by the generator notebook. Where my re-derivation and the scorecard use the same definition, the formatted numbers must be identical. Rows that this notebook does not re-run (held-out check, parameter traceability) are shown as they were reported, and are not re-checked here.

In [ ]:
SC = pd.read_csv(DATA / "validation_scorecard.csv")
mine = VER[["measure", "measured", "verdict"]].rename(columns={"measured": "re-derived here", "verdict": "verdict here"})
mine = pd.concat([mine, pd.DataFrame([{"measure": "Edge-case presence (both cases)", "re-derived here": f"{int(e1.sum()):,} and {int(e2.sum()):,} student-terms", "verdict here": "PASS" if edge_ok else "FAIL"}])], ignore_index=True)
rec = SC.merge(mine, on="measure", how="left")
rec["identical"] = np.where(rec["re-derived here"].isna(), "not re-run", np.where(rec["measured"] == rec["re-derived here"], "yes", "NO"))
banner(); display(rec[["measure", "measured", "verdict", "re-derived here", "verdict here", "identical"]].style.hide(axis="index"))
cmp_rows = rec[rec["identical"] != "not re-run"]
if FAST: print("FAST_MODE: replicate-based rows are not comparable.")
else: print(f"{(cmp_rows['identical'] == 'yes').sum()} of {len(cmp_rows)} re-derived rows are identical to the generator's scorecard.")

**What Sec. 4 establishes, and what it does not.** From the CSV files alone I reproduce the generator's reported numbers exactly: tier attrition of 67.1 % and 45.9 % (within 0.01 pp of the targets), the correlation structure (unconditional +0.39, −0.19, −0.18, against at most 0.009 once the latent state is removed), the confounding check (the naive estimate is −57.7 %, with an interval that excludes 30 %), recoverability (Mantel–Haenszel 32.2 %, Poisson 33.6 %, both with intervals covering 30 %, inside a ±8.6 pp window set by a replicate SD of 4.3 pp) and both edge cases (2,937 and 774 student-terms, well above the thresholds).

That makes the dataset a fair test bed for the analyses that follow: the confounding is real and strong, the true effect is recoverable, and the structure the whitepaper promised is present. It does **not** show that the generator resembles any real institution. Two things I did not re-run remain open and I carry them forward in Sec. 12: the held-out Class-of-2023 check, which the generator itself labels *PASS (fragile)* (only about 60 % of replicate populations land in the band), and the fact that calibration to the two tier targets is a convergence check and not independent validation.

## 5 · What does the data show? Descriptive analytics
Everything in this section is computed from the exported tables. I first build the person-period feature frame (`FE`) that Sec. 5 to Sec. 9 share. It contains one row per student-term for terms 2 to 8, because the early-warning signals need at least one prior term.

**The leakage rule for features.** A feature for the row (student *i*, term *t*) may use only (a) entry attributes and (b) values recorded in terms **1 to t−1**. Every column of the current term *t* carries the prefix `cur_` and is never used as a model input, except in one labelled contrast in Sec. 6.3. Outcome columns (`attrition_flag`, `enrollment_status`, `censored`, `dropout_flag`) and row counts (which reveal how long a student stayed) are never features.

In [ ]:
KEY = ["student_id", "term"]
PROG_MAP = {"Arts, Sciences & Social Sci.": "prog_arts_sci", "Business & Management": "prog_business", "Education": "prog_education",
            "Engineering & IT": "prog_eng_it", "Health Sciences": "prog_health", "Other": "prog_other"}

def build_features(tb):
    df = (tb["CourseGrades"].merge(tb["AttendanceRecords"], on=KEY).merge(tb["FinancialHolds"], on=KEY)
          .merge(tb["SupportInterventions"], on=KEY).merge(tb["PersistenceOutcomes"][KEY + ["dropout_flag"]], on=KEY)
          .sort_values(KEY).reset_index(drop=True))
    df["cur_grade"], df["cur_att"] = df.term_avg_score, df.attendance_rate
    df["cur_hold"], df["cur_overdue"] = df.hold_flag.astype(int), df.overdue_amount
    df["cur_passrate"] = df.units_passed / df.units_attempted
    df["cur_sub"], df["cur_sup"] = df.subsidy_status.astype(int), df.intervention.astype(int)
    g = df.groupby("student_id")
    for c in ["grade", "att", "hold", "overdue", "passrate", "sub", "sup"]:
        df[f"l1_{c}"] = g[f"cur_{c}"].shift(1)                                   # value recorded in term t-1
    df["l2_grade"] = g["cur_grade"].shift(2)
    n_prior = (df.term - 1).where(df.term > 1)
    df["prior_mean_grade"] = (g["cur_grade"].cumsum() - df.cur_grade) / n_prior  # mean over terms 1..t-1
    df["prior_mean_att"] = (g["cur_att"].cumsum() - df.cur_att) / n_prior
    df["_cm"] = g["cur_grade"].cummin(); df["prior_min_grade"] = df.groupby("student_id")["_cm"].shift(1)
    df["n_prior_holds"] = g["cur_hold"].cumsum() - df.cur_hold
    df["n_prior_support"] = g["cur_sup"].cumsum() - df.cur_sup
    df["grade_delta"] = (df.l1_grade - df.l2_grade)
    df["l1_low_g"] = (df.l1_grade < P("tau_g")).astype(int); df["l1_low_a"] = (df.l1_att < P("tau_a")).astype(int)
    df["k_l1"] = df.l1_low_g + df.l1_low_a + df.l1_hold.astype(float)
    df["cell"] = 4 * df.l1_low_g + 2 * df.l1_low_a + df.l1_hold.astype(float)
    df = df[df.term >= 2].copy()
    df["grade_delta"] = df.grade_delta.fillna(0.0)
    df["y"] = df.dropout_flag.astype(int); df["Tr"] = df.intervention.astype(bool)
    st = tb["Students"]
    ent = pd.concat([st[["student_id", "income_bracket", "distance_km"]],
                     st.working_student.astype(int).rename("working_student"), st.subsidy_at_entry.astype(int).rename("subsidy_at_entry"),
                     (st.region_tier == "high").astype(int).rename("region_high"),
                     pd.get_dummies(st.program, dtype=int).rename(columns=PROG_MAP)[sorted(PROG_MAP.values())]], axis=1)
    return df.merge(ent, on="student_id", how="left").reset_index(drop=True)

FE = build_features(TB)
assert len(FE) == len(AF) and (FE.cell.to_numpy(int) == cell).all() and (FE.y.to_numpy(bool) == y).all() and (FE.Tr.to_numpy() == Tr).all()
assert FE[["l1_grade", "l1_att", "prior_mean_grade", "prior_min_grade", "grade_delta"]].notna().all().all()
print(f"FE: {len(FE):,} student-terms (terms 2–8) from {FE.student_id.nunique():,} students | dropout events: {FE.y.sum():,} | "
      f"term-1 dropouts excluded (no prior term): {int(TB['PersistenceOutcomes'].query('term == 1').dropout_flag.sum()):,}")

In [ ]:
st = TB["Students"].set_index("student_id"); en = TB["Enrollments"]
def prof(mask_s):
    ids = st.index[mask_s]; rows_ = en.student_id.isin(ids)
    f_ = FE[FE.student_id.isin(ids)]
    return {"students": len(ids), "share working": st.loc[ids].working_student.mean(), "share with subsidy at entry": st.loc[ids].subsidy_at_entry.mean(),
            "median distance (km)": st.loc[ids].distance_km.median(), "cumulative attrition": attr_s.loc[ids].mean(),
            "mean terms observed": rows_.sum() / len(ids), "mean term grade": cg[cg.student_id.isin(ids)].term_avg_score.mean(),
            "mean attendance": ar[ar.student_id.isin(ids)].attendance_rate.mean(), "hold prevalence": fh[fh.student_id.isin(ids)].hold_flag.mean(),
            "support rate (terms 2–8)": f_.Tr.mean(), "dropout rate per term at risk (terms 2–8)": f_.y.mean()}
PROF = pd.DataFrame({"high tier": prof(st.region_tier == "high"), "typical tier": prof(st.region_tier == "typical"), "all": prof(st.region_tier.notna())})
fmt = {i: ("{:,.0f}" if i == "students" else "{:.2f}" if i in ("mean terms observed", "median distance (km)", "mean term grade") else "{:.1%}") for i in PROF.index}
banner(); display(PROF.T.style.format({c: fmt[c] for c in PROF.index}))

In [ ]:
# per-term hazard and cumulative attrition by tier, from the exports
po_t = PO.merge(st[["region_tier"]], left_on="student_id", right_index=True)
haz = po_t.groupby(["region_tier", "term"]).dropout_flag.agg(["sum", "size"]).rename(columns={"sum": "events", "size": "at risk"}).assign(hazard=lambda x: x.events / x["at risk"])
wd_term = en[en.enrollment_status == "withdrawn"].set_index("student_id").term
cum = {t_: [(wd_term.reindex(st.index[st.region_tier == t_]).le(k_).sum() / (st.region_tier == t_).sum()) for k_ in range(1, T + 1)] for t_ in ("high", "typical")}
inc_work = (attr_s.to_frame("attr").join(st[["income_bracket", "working_student"]]).groupby(["income_bracket", "working_student"]).attr.mean().unstack())

fig, ax = plt.subplots(2, 2, figsize=(13, 9)); ax = ax.ravel()
for t_, col, tgt in [("high", "#c0392b", P("target_high_2022")), ("typical", "#2471a3", P("target_typical_2022"))]:
    ax[0].plot(range(1, T + 1), cum[t_], marker="o", color=col, label=t_); ax[0].axhline(tgt, color=col, ls=":", lw=1)
    ax[1].plot(range(1, T + 1), haz.loc[t_].hazard, marker="o", color=col, label=t_)
ax[0].set(title="(a) Cumulative attrition by term (dotted = calibration target)", xlabel="term", ylabel="share of students withdrawn"); ax[0].legend()
ax[1].set(title="(b) Dropout rate among students still enrolled, by term", xlabel="term", ylabel="dropout rate per term at risk"); ax[1].legend()
w = 0.38; xb = np.arange(1, 6)
ax[2].bar(xb - w/2, inc_work[False], w, color="#7fb3d5", label="not working"); ax[2].bar(xb + w/2, inc_work[True], w, color="#1f618d", label="working student")
ax[2].set(title="(c) Cumulative attrition by income bracket (1 = lowest) and work status", xlabel="income bracket", ylabel="share withdrawn", xticks=xb); ax[2].legend()
for lab_, col in [(0, "#2e86c1"), (1, "#c0392b")]:
    ax[3].hist(FE.loc[FE.y == lab_, "l1_grade"], bins=40, density=True, alpha=.55, color=col, label="dropped out this term" if lab_ else "stayed")
ax[3].axvline(P("tau_g"), color="k", ls="--", lw=1); ax[3].text(P("tau_g") + .5, ax[3].get_ylim()[1] * .9, "referral cut-off", fontsize=8)
ax[3].set(title="(d) Previous-term grade, by whether the student drops out now", xlabel="term t−1 average score", ylabel="density"); ax[3].legend()
for a_ in ax: a_.grid(alpha=.25)
savefig(fig, "fig1_descriptive.png", "Descriptive analytics from the exported tables"); plt.show()

**Reading Figure 1.** Cumulative attrition ends at the calibrated 67.1 % and 45.9 % by construction (panel a). The high tier loses students faster at every term (panel b: 15.4 % against 8.5 % in term 1, 12.1 % against 6.6 % in term 8). The per-term rate *falls* with time in both tiers, which looks like retention getting easier but is selection: the most vulnerable students leave first, so those remaining are a less vulnerable group. Panel c shows attrition ranging from about 35 % (non-working students in the highest income bracket) to about 67 % (working students in the lowest). That gradient is built into the generator through the vulnerability equation, so I read it as a check that the structure is present and not as a finding about the world. Panel d shows why prediction is hard: students who drop out have a lower previous-term grade on average, but the two distributions overlap heavily, and the referral cut-off at 60 separates them poorly.

## 6 · Can attrition be predicted before it happens? An early-warning model
**Task.** For each student still enrolled at the start of term *t* (*t* = 2 to 8), predict whether they withdraw in term *t*, using only what was known at the end of term *t*−1. This is a discrete-time hazard problem in person-period form (Gelman, Hill & Vehtari 2020, in the survival-as-binary-regression sense).

**Choices I made, and why.**
* **Features:** entry attributes plus prior-term records (Sec. 5). I *exclude* the regional tier from the main model. In the generator the tier is just an entry attribute, but flagging students for support on the basis of the region they come from is the kind of use WP Sec. 13 warns against, so the default model does not get it. Sec. 6.3 shows what including it would change.
* **Validation:** 5-fold cross-validation **grouped by student**, so a student's rows never sit on both sides of a split. Every model sees the same folds, so differences between models are paired. Hyperparameters are fixed and not tuned (`ASSUMED`, D3 registry).
* **Models:** a **discrete-time logistic hazard** (interpretable, with a dummy per term), a **gradient-boosted tree** (flexible), and the **rule-based baseline** an institution would already have: how many of the three referral signals fired last term.
* **Metrics.** AUC for ranking. PR-AUC and capture at fixed review capacity (top 5 %, 10 % and 20 % of student-terms) because the event is rare, and capacity is what a programme actually has. Brier score and a reliability table for calibration.
* **What a prediction here means.** The outcome is dropout *under the current support practice*: supported student-terms had their hazard cut by 30 %. A model trained on this data therefore estimates risk-as-currently-managed, not risk-if-left-alone. Sec. 7 and Sec. 9 come back to this.

### 6.1 Features, folds and models

In [ ]:
F_ENTRY = ["income_bracket", "working_student", "distance_km", "subsidy_at_entry"] + sorted(PROG_MAP.values())
F_PRIOR = ["term", "l1_grade", "l1_att", "l1_hold", "l1_overdue", "l1_passrate", "l1_sub", "l1_sup", "l1_low_g", "l1_low_a", "k_l1",
           "prior_mean_grade", "prior_mean_att", "prior_min_grade", "n_prior_holds", "n_prior_support", "grade_delta"]
F_HGB = F_ENTRY + F_PRIOR
F_LOGIT = [c for c in F_ENTRY if c != "prog_arts_sci"] + [c for c in F_PRIOR if c != "term"]       # one programme dummy dropped as the reference
XH = FE[F_HGB]
XL = pd.concat([FE[F_LOGIT], pd.get_dummies(FE.term, prefix="term", dtype=int).drop(columns="term_2")], axis=1)

FORBIDDEN = {"attrition_flag", "enrollment_status", "censored", "dropout_flag", "y", "Tr", "intervention", "risk_score", "cell", "student_id", "region_high"}
def leakage_guard(cols, allow_current=False):
    bad = [c for c in cols if c in FORBIDDEN or (c.startswith("cur_") and not allow_current)]
    assert not bad, f"forbidden feature(s): {bad}"
leakage_guard(F_HGB); leakage_guard(list(XL.columns))
print(f"Main-model features: {len(F_HGB)} (gradient boosting), {XL.shape[1]} (logistic, with term dummies). Leakage guard passed: nothing from term t or later, no outcome columns, no region.")

yv = FE.y.to_numpy(); groups = FE.student_id.to_numpy()
FOLDS = list(GroupKFold(n_splits=Q("cv_folds"), shuffle=True, random_state=Q("cv_seed")).split(XH, yv, groups))
assert all(not (set(groups[a]) & set(groups[b])) for a, b in FOLDS)
print(f"{Q('cv_folds')} folds, grouped by student; events per fold: {[int(yv[b].sum()) for _, b in FOLDS]}")

make_logit = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=Q("logit_C"), max_iter=2000))
make_hgb = lambda: HistGradientBoostingClassifier(**Q("hgb_params"))
def oof_predict(make, X):
    oof, aucs = np.zeros(len(yv)), []
    for a, b in FOLDS:
        m = make().fit(X.iloc[a], yv[a]); oof[b] = m.predict_proba(X.iloc[b])[:, 1]; aucs.append(roc_auc_score(yv[b], oof[b]))
    return oof, np.array(aucs)

t0 = time.time(); OOF, FA = {}, {}
OOF["logit"], FA["logit"] = oof_predict(make_logit, XL)
OOF["hgb"], FA["hgb"] = oof_predict(make_hgb, XH)
rule_rng = np.random.default_rng(Q("cv_seed"))
OOF["rule"] = FE.k_l1.to_numpy(float) + rule_rng.uniform(0, 1e-6, len(FE))       # tiny jitter only breaks ties for the capture curves
FA["rule"] = np.array([roc_auc_score(yv[b], OOF["rule"][b]) for _, b in FOLDS])
print(f"fitted in {time.time()-t0:.0f}s")

### 6.2 Results (out-of-fold)

In [ ]:
def capture_curve(y_, s_):
    o = np.argsort(-s_, kind="stable"); return np.cumsum(y_[o]) / y_.sum()
def metric_row(name, s_, aucs=None, prob=True):
    out = {"model": name, "AUC": roc_auc_score(yv, s_), "AUC SD across folds": (aucs.std(ddof=1) if aucs is not None else np.nan),
           "PR-AUC": average_precision_score(yv, s_), "Brier": (brier_score_loss(yv, s_) if prob else np.nan)}
    cc = capture_curve(yv, s_)
    for q in Q("topk_shares"): out[f"capture top {q:.0%}"] = cc[int(round(q * len(yv))) - 1]
    return out
base_rate = yv.mean()
RES = pd.DataFrame([metric_row("Rule: count of prior-term signals fired", OOF["rule"], FA["rule"], prob=False),
                    metric_row("Logistic hazard (discrete-time)", OOF["logit"], FA["logit"]),
                    metric_row("Gradient-boosted trees", OOF["hgb"], FA["hgb"])])
banner()
print(f"Out-of-fold results on {len(FE):,} student-terms; event rate {base_rate:.2%}; Brier of a constant-rate forecast = {base_rate*(1-base_rate):.4f}; random capture = the review share itself")
display(RES.style.format({c: "{:.3f}" for c in ["AUC", "AUC SD across folds", "PR-AUC"]} | {"Brier": "{:.4f}"} | {c: "{:.1%}" for c in RES.columns if c.startswith("capture")}, na_rep="n/a").hide(axis="index"))

### 6.3 Two contrasts: a model I should *not* deploy, and a feature I chose not to use
* **Leaky contrast (not deployable).** The same boosted trees with the *current-term* grade, attendance and hold added. These are recorded in the term the student drops out (or not), and they carry that term's shock, so they are not available when the early warning has to be raised. I include this model only to show how much apparent skill comes from information the programme cannot have in time.
* **With regional tier.** The main model plus the tier flag, to show what I gave up by leaving it out.

In [ ]:
XHleak = pd.concat([XH, FE[["cur_grade", "cur_att", "cur_hold"]]], axis=1)
XHreg = pd.concat([XH, FE[["region_high"]]], axis=1)
leakage_guard(list(XHleak.columns), allow_current=True)                  # the contrast is the only place current-term columns are allowed
OOF["hgb_leaky"], FA["hgb_leaky"] = oof_predict(make_hgb, XHleak)
OOF["hgb_region"], FA["hgb_region"] = oof_predict(make_hgb, XHreg)
CON = pd.DataFrame([metric_row("Boosted trees + current-term signals  [LEAKY CONTRAST, not deployable]", OOF["hgb_leaky"], FA["hgb_leaky"]),
                    metric_row("Boosted trees + regional tier  [ethics ablation]", OOF["hgb_region"], FA["hgb_region"]),
                    metric_row("Boosted trees (main model, for reference)", OOF["hgb"], FA["hgb"])])
banner(); display(CON.style.format({c: "{:.3f}" for c in ["AUC", "AUC SD across folds", "PR-AUC"]} | {"Brier": "{:.4f}"} | {c: "{:.1%}" for c in CON.columns if c.startswith("capture")}).hide(axis="index"))

### 6.4 How good could any model be? Ceilings from the hidden state *(uses the hidden state to grade the model)*
A model's AUC means little without a ceiling. Because I know the true hazard, I can compute two reference points on exactly the same student-terms:
* **Ceiling A, the true hazard.** The probability that the generator actually used, which includes baseline vulnerability, *this* term's shock and *this* term's support. No predictor, however good its data, can rank better in expectation.
* **Ceiling B, baseline vulnerability but no shock.** What a predictor would achieve if it knew each student's latent baseline vulnerability exactly but could not foresee the shock (shocks are independent of everything before them, so nobody can). It ignores this term's support, so it is a reference point and not a strict bound.

In [ ]:
CEIL = None
if GT_OK:
    gt = FE[KEY].merge(AUD["panel"][KEY + ["hazard"]], on=KEY, how="left").merge(AUD["student"], on="student_id", how="left")
    assert len(gt) == len(FE) and gt.hazard.notna().all() and gt.baseline_vulnerability.notna().all()
    h_true = gt.hazard.to_numpy(); ps = P("shock_prob"); vb = gt.baseline_vulnerability.to_numpy()
    p_vonly = (1 - ps) * sig(P("gamma0") + P("gamma1") * vb) + ps * sig(P("gamma0") + P("gamma1") * vb + P("gamma2"))
    CEIL = {"A": h_true, "B": p_vonly}
    cdf = pd.DataFrame([metric_row("Ceiling A: true hazard (knows v, this term's shock and support)", h_true),
                        metric_row("Ceiling B: knows baseline vulnerability only", p_vonly)]).drop(columns=["AUC SD across folds", "Brier"])
    banner(); display(cdf.style.format({"AUC": "{:.3f}", "PR-AUC": "{:.3f}"} | {c: "{:.1%}" for c in cdf.columns if c.startswith("capture")}).hide(axis="index"))
    a_h = RES.loc[RES.model.str.startswith("Gradient"), "AUC"].item(); a_r = RES.loc[RES.model.str.startswith("Rule"), "AUC"].item()
    a_A, a_B = cdf.AUC.iloc[0], cdf.AUC.iloc[1]
    print(f"AUC: rule {a_r:.3f} → boosted trees {a_h:.3f} → ceiling B {a_B:.3f} → ceiling A {a_A:.3f}.")
    print(f"The boosted model closes {(a_h - a_r) / (a_B - a_r):.0%} of the gap between the rule and ceiling B, and {(a_h - a_r) / (a_A - a_r):.0%} of the gap to ceiling A.")
else:
    print("Skipped: exact reproduction of the exported dataset is not confirmed on this platform (see Sec. 3).")

### 6.5 Summary figure and calibration

In [ ]:
# ---- figure 2: model comparison, capture, calibration, drivers -----------------------------------------------
fig, ax = plt.subplots(2, 2, figsize=(13, 9)); ax = ax.ravel()
names = [("rule", "rule\n(signal count)", "#95a5a6"), ("logit", "logistic\nhazard", "#2471a3"), ("hgb", "boosted\ntrees", "#1e8449")]
vals = [roc_auc_score(yv, OOF[k_]) for k_, _, _ in names]; errs = [FA[k_].std(ddof=1) for k_, _, _ in names]
labels = [l_ for _, l_, _ in names]; cols_ = [c_ for _, _, c_ in names]
vals.append(roc_auc_score(yv, OOF["hgb_leaky"])); errs.append(FA["hgb_leaky"].std(ddof=1)); labels.append("leaky\n(not deployable)"); cols_.append("#c0392b")
if CEIL is not None:
    for k_, l_ in [("B", "ceiling B\n(knows v)"), ("A", "ceiling A\n(true hazard)")]: vals.append(roc_auc_score(yv, CEIL[k_])); errs.append(0); labels.append(l_); cols_.append("#7d3c98")
ax[0].bar(range(len(vals)), vals, yerr=errs, color=cols_, capsize=3); ax[0].set_xticks(range(len(vals))); ax[0].set_xticklabels(labels, fontsize=8)
ax[0].set(title="(a) Out-of-fold AUC (error bar = SD over folds)", ylim=(0.5, max(vals) + 0.04), ylabel="AUC"); ax[0].axhline(0.5, color="k", lw=.8)
for i_, v_ in enumerate(vals): ax[0].text(i_, v_ + 0.006, f"{v_:.3f}", ha="center", fontsize=8)
share = np.arange(1, len(yv) + 1) / len(yv)
for k_, l_, c_ in [("rule", "rule", "#95a5a6"), ("logit", "logistic hazard", "#2471a3"), ("hgb", "boosted trees", "#1e8449")]:
    ax[1].plot(share, capture_curve(yv, OOF[k_]), color=c_, label=l_)
if CEIL is not None:
    ax[1].plot(share, capture_curve(yv, CEIL["B"]), color="#7d3c98", ls="--", label="ceiling B"); ax[1].plot(share, capture_curve(yv, CEIL["A"]), color="#7d3c98", ls=":", label="ceiling A")
ax[1].plot([0, 1], [0, 1], color="k", lw=.8, label="random"); ax[1].set_xlim(0, 0.4); ax[1].set_ylim(0, 0.8)
ax[1].set(title="(b) Share of dropouts captured when reviewing the top x of student-terms", xlabel="share of student-terms reviewed", ylabel="share of dropout events captured"); ax[1].legend(fontsize=8)
for k_, c_ in [("logit", "#2471a3"), ("hgb", "#1e8449")]:
    dec = pd.qcut(pd.Series(OOF[k_]).rank(method="first"), 10, labels=False)
    cal_ = pd.DataFrame({"p": OOF[k_], "y": yv, "d": dec}).groupby("d").mean(); ax[2].plot(cal_.p, cal_.y, marker="o", color=c_, label=k_)
mx = max(cal_.p.max(), cal_.y.max()) * 1.05; ax[2].plot([0, mx], [0, mx], color="k", lw=.8)
ax[2].set(title="(c) Reliability: predicted vs observed dropout rate, by decile", xlabel="mean predicted risk", ylabel="observed dropout rate"); ax[2].legend()
a_f, b_f = FOLDS[0]; m_f = make_hgb().fit(XH.iloc[a_f], yv[a_f])
pi_ = permutation_importance(m_f, XH.iloc[b_f], yv[b_f], scoring="roc_auc", n_repeats=5, random_state=Q("cv_seed"), n_jobs=1)
imp = pd.Series(pi_.importances_mean, index=XH.columns).sort_values(ascending=False).head(10)[::-1]
ax[3].barh(imp.index, imp.values, xerr=pd.Series(pi_.importances_std, index=XH.columns).reindex(imp.index).values, color="#1e8449", capsize=2)
ax[3].set(title="(d) Boosted trees: permutation importance (AUC lost when a feature is shuffled)", xlabel="drop in AUC on held-out fold 1")
for a_ in ax: a_.grid(alpha=.25)
savefig(fig, "fig2_prediction.png", "Early-warning model: ranking, capture, calibration and drivers"); plt.show()

In [ ]:
# reliability table for the main model, and the share of events a model sees before the first signal can exist
dec = pd.qcut(pd.Series(OOF["hgb"]).rank(method="first"), 10, labels=False) + 1
rel_t = pd.DataFrame({"p": OOF["hgb"], "y": yv, "d": dec}).groupby("d").agg(student_terms=("y", "size"), mean_predicted=("p", "mean"), observed=("y", "mean"))
banner(); print("Reliability of the boosted-tree model by risk decile (1 = lowest predicted risk):"); display(rel_t.style.format({"student_terms": "{:,}", "mean_predicted": "{:.3f}", "observed": "{:.3f}"}))
n1 = int(TB["PersistenceOutcomes"].query("term == 1").dropout_flag.sum()); nall = int(TB["PersistenceOutcomes"].dropout_flag.sum())
print(f"{n1:,} of {nall:,} dropouts ({n1/nall:.1%}) happen in term 1, before any prior-term record exists, so no term-based early warning can see them.")

**Reading the early-warning results.**
* **The model ranks better than the rule, and the gain is real but modest.** AUC 0.655 against 0.567, PR-AUC 0.140 against a base rate of 8.0 %, and a top-10 % capture of 21.4 % against 15.3 % for the rule (a lift of about 2.1). The SD across folds is 0.005, so the gap between the model and the rule is far outside fold noise. That SD does not include the uncertainty from drawing one particular population, which I did not measure for the model.
* **The logistic hazard and the boosted trees are indistinguishable** (0.654 against 0.655). The signal in this dataset is close to additive, so extra flexibility buys nothing. The reliability table shows both are well calibrated across all ten deciles.
* **The ceilings put the numbers in context.** The model closes 76 % of the gap between the rule and what a predictor could achieve if it knew every student's baseline vulnerability but not the shocks (ceiling B, 0.683). Even the true hazard gives only 0.722 (ceiling A), because this term's shock and the support decision are not knowable in advance. So in this synthetic world a large share of dropout is simply unforeseeable.
* **The leaky contrast is worth the 0.025 it adds, and I cannot use it.** With the current term's grade, attendance and hold the AUC reaches 0.680, because those values carry that term's shock. They do not exist when the warning has to be raised.
* **Leaving out the regional tier costs nothing here** (0.656 with it, 0.655 without). The entry attributes and prior-term records already carry what the tier would add, so the ethical choice of Sec. 6 has no accuracy cost in this dataset.
* **What drives the model is the running average of the record, not last term's flags.** Shuffling the prior mean grade costs 0.054 AUC and shuffling the prior mean attendance 0.032. Every other feature costs less than 0.003 (Figure 2d). Averages over several terms are a better proxy for the hidden baseline vulnerability than a single term's binary flag, which is also why the rule, built on last term's flags, ranks poorly. I use permutation importance and not regression coefficients, because the features are strongly correlated and individual coefficients are not interpretable.
* **A fifth of dropouts cannot be seen by any term-based model.** 19.4 % occur in term 1. That group needs entry-time information and a different tool.

## 7 · Does good prediction tell me whether support works? The actionability gap
A reasonable-looking dashboard puts a risk model next to the support programme and asks: *the students we support are the high-risk ones, so does the support help?* A prediction model cannot answer that. It ranks students by risk; it says nothing about what would have happened to them without support. The review literature's practitioner recommendation (Lopez-Muñoz et al. 2026) is that models serve as decision-support tools and not as stand-alone answers. This section shows what goes wrong when a model is stretched into a verdict on the programme.

I compare five readings of the *same* data on the *same* student-terms, with the known truth of a 30 % reduction in the hazard.

In [ ]:
sc = OOF["hgb"]; Trv = FE.Tr.to_numpy(); cap = int(Trv.sum())
top_flag = sc >= np.sort(sc)[-cap]                                       # the cap highest-risk student-terms, same capacity as the actual programme
def naive_red(mask): return rel(yv[mask & Trv].mean() / yv[mask & ~Trv].mean())
octile = pd.qcut(pd.Series(sc).rank(method="first"), 8, labels=False).to_numpy()
mh_o, ci_o = mh_rr(yv.astype(bool), Trv, octile)
rows = [("1. Ranking quality of the model", f"AUC {roc_auc_score(yv, sc):.3f}", "a good ranker says nothing about the programme"),
        ("2. Raw comparison, supported vs not supported", f"{EM['naive']:+.1%}", "negative = supported students dropped out MORE often: the programme looks harmful"),
        (f"3. Raw comparison inside the model's top {cap:,} highest-risk terms", f"{naive_red(top_flag):+.1%}", "the sign improves, but the size is far from the truth: bias reduced, not removed"),
        ("4. Stratify on the model's own risk score (8 strata)", f"{rel(mh_o):+.1%} (95% CI {rel(ci_o[1]):+.1%} to {rel(ci_o[0]):+.1%})", "score strata are not the assignment strata, so the interval can miss the truth (compare with the last row)"),
        ("5. Stratify on the assignment signals (8 cells), Mantel–Haenszel", f"{EM['mh']:+.1%} (95% CI {CI['mh'][0]:+.1%} to {CI['mh'][1]:+.1%})", "the right adjustment set: its interval covers the truth"),
        ("Truth (by construction)", f"{TRUE_EFFECT:+.0%}", "")]
GAP = pd.DataFrame(rows, columns=["reading of the data", "estimated relative reduction in dropout", "what it tells a programme manager"])
banner(); display(GAP.style.hide(axis="index").set_properties(**{"text-align": "left"}))

In [ ]:
oc = pd.DataFrame({"octile": octile + 1, "y": yv, "Tr": Trv, "p": sc})
og = oc.groupby("octile").agg(student_terms=("y", "size"), mean_predicted=("p", "mean"), share_supported=("Tr", "mean"))
og["dropout rate, supported"] = oc[oc.Tr].groupby("octile").y.mean(); og["dropout rate, not supported"] = oc[~oc.Tr].groupby("octile").y.mean()
og["supported n"] = oc[oc.Tr].groupby("octile").size(); og["not supported n"] = oc[~oc.Tr].groupby("octile").size()
banner(); print("Dropout rates by the model's risk octile (1 = lowest). Support is concentrated at the very top, where both arms are thin:")
display(og.style.format({"student_terms": "{:,}", "mean_predicted": "{:.3f}", "share_supported": "{:.1%}", "dropout rate, supported": "{:.1%}", "dropout rate, not supported": "{:.1%}", "supported n": "{:,.0f}", "not supported n": "{:,.0f}"}, na_rep="none"))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
labs = ["raw\n(all terms)", "raw\n(model's top\nflagged)", "stratified on\nmodel score", "stratified on\nassignment signals", "truth"]
vals = [EM["naive"], naive_red(top_flag), rel(mh_o), EM["mh"], TRUE_EFFECT]; colr = ["#c0392b", "#c0392b", "#e67e22", "#1e8449", "k"]
ax[0].bar(range(5), vals, color=colr); ax[0].axhline(0, color="k", lw=.8); ax[0].set_xticks(range(5)); ax[0].set_xticklabels(labs, fontsize=8)
for i_, v_ in enumerate(vals): ax[0].text(i_, (v_ + 0.02) if v_ >= 0 else (v_ / 2), f"{v_:+.0%}", ha="center", va="bottom" if v_ >= 0 else "center", fontsize=9, color="k" if v_ >= 0 else "w")
ax[0].set_ylim(-0.68, 0.45)
ax[0].set(title="(a) Estimated relative reduction in dropout from the support programme", ylabel="relative reduction (negative = support looks harmful)")
w = 0.38; xo = og.index.to_numpy()
ax[1].bar(xo - w/2, og["dropout rate, not supported"], w, color="#85929e", label="not supported"); ax[1].bar(xo + w/2, og["dropout rate, supported"], w, color="#c0392b", label="supported")
ax[1].set(title="(b) Dropout rate by model risk octile and support status", xlabel="model risk octile (8 = highest)", ylabel="dropout rate", xticks=xo); ax[1].legend()
for a_ in ax: a_.grid(alpha=.25)
savefig(fig, "fig3_actionability_gap.png", "Prediction is not a verdict on the programme"); plt.show()

**Reading the gap.** A model with a perfectly respectable AUC leads to the wrong decision if it is used to judge the programme.
* The raw comparison (−57.7 %) says the programme *increases* dropout by more than half. This is confounding by indication: the programme supports exactly the students who are about to be most at risk, so supported students drop out more often *despite* the 30 % benefit.
* Restricting to the model's highest-risk terms moves the estimate to +11 %, and stratifying on the model's risk score gives +7.4 % (95 % CI +3.0 % to +11.7 %). The sign is now right, but both are far below the truth, and the second interval excludes 30 %. The likely reason is that score strata are not the assignment strata, so within a stratum supported and unsupported student-terms still differ in which signals fired. I did not investigate the mechanism further, so I state it as the likely cause and not as a finding.
* Only stratifying on the signals the programme actually used (Mantel–Haenszel on the eight cells) recovers the effect: 32.2 % with an interval covering 30 %.
* The prediction question ("who is likely to drop out?") and the evaluation question ("did our support change that?") need different designs. A risk score is useful for allocating attention. It is not evidence about whether the attention helped.

## 8 · Does the support programme work? Causal analysis against the known 30 %
**The question is causal, so the first step is the graph, not the model.** Figure 4 is my reading of the generator's structure (WP Fig. 2). Baseline vulnerability *v* drives the three signals in every term and also drives dropout. The programme assigns support at term *t* using the signals recorded at *t*−1. So *v* is a **common cause** of support and dropout: it opens the backdoor path *support ← signals(t−1) ← v → dropout*. Conditioning on the observed signals at *t*−1 blocks that path (Peters, Janzing & Schölkopf 2017), and because the assignment probability depends *only* on those signals, support is as good as random within a signal cell. That is the ignorability condition I rely on (Gelman, Hill & Vehtari 2020). The shock at *t* is independent of everything before it, so it cannot confound.

Three things have to hold for the adjusted estimates to be trusted, and I check each: **overlap** (both arms exist in the cells I stratify on, Sec. 8.2), **the right adjustment set** (Sec. 8.3) and **estimator stability** (Sec. 8.1 and Sec. 8.2).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.2)); ax.set_axis_off(); ax.set_xlim(0, 12); ax.set_ylim(0, 5.6)
nodes = {"v": (1.4, 2.8, "baseline\nvulnerability  v\n(latent)", True), "O1": (4.7, 4.5, "signals at t−1\ngrade · attendance · hold", False),
         "Tt": (8.0, 4.5, "support at t", False), "D": (10.6, 2.8, "dropout at t", False),
         "S": (8.0, 1.0, "shock at t\n(latent)", True), "O2": (4.7, 1.0, "signals at t\n(not available in time)", False)}
tx = {}
for k_, (x_, y_, s_, lat) in nodes.items():
    tx[k_] = ax.text(x_, y_, s_, ha="center", va="center", fontsize=10, zorder=3,
                     bbox=dict(boxstyle="round,pad=0.45", fc="#f4f6f7" if not lat else "white", ec="#7f8c8d", ls="--" if lat else "-", lw=1.4))
def arrow(a, b, color="#2c3e50", ls="-", lw=1.6, label=None, rad=0.0, lpos=0.5, dy=0.18):
    ax.annotate("", xy=nodes[b][:2], xytext=nodes[a][:2], zorder=2,
                arrowprops=dict(arrowstyle="-|>", color=color, ls=ls, lw=lw, shrinkA=2, shrinkB=2, patchA=tx[a].get_bbox_patch(), patchB=tx[b].get_bbox_patch(), connectionstyle=f"arc3,rad={rad}"))
    if label:
        xm = nodes[a][0] + lpos * (nodes[b][0] - nodes[a][0]); ym = nodes[a][1] + lpos * (nodes[b][1] - nodes[a][1]) + dy
        ax.text(xm, ym, label, fontsize=8, ha="center", color=color)
arrow("v", "O1", "#c0392b", lw=2.2); arrow("v", "D", "#c0392b", lw=2.2, rad=0.0, lpos=0.5, dy=-0.55, label="v raises dropout risk")
arrow("O1", "Tt", label="assignment rule (Eq. 1–2)", dy=0.2); arrow("Tt", "D", "#1e8449", lw=2.2); ax.text(9.55, 4.15, "true effect:\nhazard × 0.70", fontsize=8, ha="left", color="#1e8449")
arrow("S", "D"); arrow("S", "O2"); arrow("v", "O2", "#7f8c8d", ls=":")
ax.text(6.0, 5.35, "Backdoor path (red): support ← signals(t−1) ← v → dropout. Stratifying on signals(t−1) blocks it.", fontsize=9, ha="center", color="#c0392b")
ax.text(6.0, 0.08, "The shock at t is independent of everything before t, so it cannot confound the support effect.", fontsize=9, ha="center", color="#34495e")
savefig(fig, "fig4_dag.png", "Causal structure of the support programme"); plt.show()

### 8.1 Estimates on the exported data, and across replicate populations
All eight estimators, run on the main population (from the exported CSV files) and on the 20 replicate populations from Sec. 4.5. The replicate columns are what tell me whether one population's answer can be trusted: an estimator that lands near 30 % in one population but swings across populations (as inverse-probability weighting does) is not a dependable tool, whatever it happens to give once. IPW here uses the **true** propensity (the exported `risk_score`), which is the *best* case; an analyst would have to estimate it.

In [ ]:
EST = pd.DataFrame({
    "estimator": [ESTLAB[k_] for k_ in ESTLAB],
    "main population": [EM[k_] for k_ in ESTLAB],
    "95% CI": [(f"{CI[k_][0]:+.1%} to {CI[k_][1]:+.1%}" if k_ in CI else "n/a") for k_ in ESTLAB],
    "replicate mean": [REP[k_].mean() for k_ in ESTLAB], "replicate SD": [REP[k_].std(ddof=1) for k_ in ESTLAB],
    "replicate min": [REP[k_].min() for k_ in ESTLAB], "replicate max": [REP[k_].max() for k_ in ESTLAB]})
EST["bias (mean − 30%)"] = EST["replicate mean"] - TRUE_EFFECT
banner(); print(f"Relative reduction in dropout (1 − ratio); truth = {TRUE_EFFECT:.0%}; {len(REP)} replicate populations")
display(EST.style.format({c: "{:+.1%}" for c in EST.columns if c not in ("estimator", "95% CI")}).hide(axis="index"))

**Reading the estimators.**
* **Confounding is structural, not noise.** The naive estimate is −57.7 % on the main population and between −64.9 % and −52.3 % across all 20 populations.
* **Mantel–Haenszel and Poisson recover the effect.** On the main population they give 32.2 % and 33.6 %. Across 20 populations the means are 28.9 % and 29.0 % with SD 4.3 and 4.2 points, and every population lands between 20 % and 37 %. The practical meaning: at this cohort size, one evaluation pins the effect down only to roughly ±8 points. The small downward bias (about 1 point) is within what 20 replicates can resolve.
* **IPW is unreliable here, and one untrimmed number that looks right is luck.** Untrimmed Hajek and Horvitz–Thompson have replicate SDs of 37 and 38 points and ranges from about −47 % or −52 % to +72 % or +73 %. The main population's untrimmed Horvitz–Thompson value (+32.7 %) happens to sit near the truth, but its replicate SD says nothing should be read into that. The clipped variants are stable but wrong in opposite directions (Hajek −15.1 % on average, Horvitz–Thompson +73.3 %), so the answer depends on an implementation choice. This is the best case for IPW, since I give it the *true* propensity.
* **The logistic odds ratio is close but is the wrong quantity.** It averages +32.0 % across populations, an odds ratio of about 0.68 against a risk ratio of 0.70. The gap is small at a baseline hazard near 10 %. The whitepaper (Sec. 11) expects it to widen when the baseline hazard is higher. I did not test that here.

### 8.2 Overlap and the weights that make IPW unstable
The assignment probability is a steep step in the number of signals fired (`β0 = −9`, `β = 6`, WP Table 4): about 0.0001 with no signal, 0.05 with one, 0.95 with two, and about 1 with three. Only cells with one or two signals contain both arms. The table that follows shows what that does to inverse-probability weights.

In [ ]:
w_t = np.where(Tr, 1 / np.clip(prop, 1e-12, 1), 1 / np.clip(1 - prop, 1e-12, 1))
ess = lambda w_: w_.sum() ** 2 / (w_ ** 2).sum()
ipw_diag = pd.DataFrame([{"arm": a_, "student-terms": int(m_.sum()), "effective sample size": ess(w_t[m_]), "max weight": w_t[m_].max(),
                          "share of weight in top 10 terms": np.sort(w_t[m_])[-10:].sum() / w_t[m_].sum()} for a_, m_ in [("supported", Tr), ("not supported", ~Tr)]])
banner(); display(ipw_diag.style.format({"student-terms": "{:,}", "effective sample size": "{:,.0f}", "max weight": "{:,.0f}", "share of weight in top 10 terms": "{:.1%}"}).hide(axis="index"))

The weights explain the instability. In the supported arm, 15,293 student-terms carry an effective sample size of 43, the largest single weight is 8,104, and ten student-terms hold about 30 % of the total weight. A handful of unusual supported terms (those with a weak signal count but, by chance, selected) decide the estimate.

### 8.3 Which adjustment set? What happens if the analyst controls for the wrong things
The Mantel–Haenszel estimate stratifies on all three signals. An analyst who only logs *some* of the referral signals (for example the grade flag but not the hold) leaves part of the backdoor path open. The table shows the cost, and the last two rows show that the count of signals alone is enough (it is a function of the propensity score, which is a balancing score), but that grade alone is not.

In [ ]:
lg_, la_, lh_ = (cell >> 2) & 1, (cell >> 1) & 1, cell & 1
sets = [("none (raw comparison)", np.zeros_like(cell)), ("grade signal only", lg_), ("attendance signal only", la_), ("hold only", lh_),
        ("grade + attendance", 2 * lg_ + la_), ("grade + hold", 2 * lg_ + lh_), ("attendance + hold", 2 * la_ + lh_),
        ("all three signals (8 cells)", cell), ("count of signals fired (0–3)", kk)]
adj = []
for nm, code_ in sets:
    r_, c_ = mh_rr(y, Tr, np.asarray(code_, dtype=int)); adj.append({"adjustment set": nm, "estimate": rel(r_), "lo": rel(c_[1]), "hi": rel(c_[0])})
ADJ = pd.DataFrame(adj); ADJ["covers 30%"] = (ADJ.lo <= TRUE_EFFECT) & (TRUE_EFFECT <= ADJ.hi)
banner(); display(ADJ.style.format({"estimate": "{:+.1%}", "lo": "{:+.1%}", "hi": "{:+.1%}"}).hide(axis="index"))

zero = np.zeros(len(y), dtype=int); names_c = {1: "hold", 2: "attendance", 3: "att.+hold", 4: "grade", 5: "grade+hold", 6: "grade+att.", 7: "all three"}
cc_rows = []
for c_ in range(1, 8):
    m_ = cell == c_; nt, nu = int((m_ & Tr).sum()), int((m_ & ~Tr).sum())
    if nt >= Q("min_arm") and nu >= Q("min_arm"):
        r_, ci_ = mh_rr(y[m_], Tr[m_], zero[m_]); cc_rows.append({"cell": names_c[c_], "signals": bin(c_).count("1"), "supported": nt, "not supported": nu, "estimate": rel(r_), "lo": rel(ci_[1]), "hi": rel(ci_[0])})
CC = pd.DataFrame(cc_rows)
print(f"Cell-by-cell estimates (cells with ≥ {Q('min_arm')} student-terms in each arm). The true effect is the same 30% in every cell by construction:")
display(CC.style.format({"supported": "{:,}", "not supported": "{:,}", "estimate": "{:+.1%}", "lo": "{:+.1%}", "hi": "{:+.1%}"}).hide(axis="index"))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9.2)); ax = ax.ravel()
ek = list(ESTLAB); good = {"mh", "poisson"}; ypos = np.arange(len(ek))[::-1]
ax[0].axvline(TRUE_EFFECT, color="k", ls="--", lw=1.2, label="true effect 30%")
for yy_, k_ in zip(ypos, ek):
    c_ = "#1e8449" if k_ in good else "#c0392b"; ax[0].scatter(EM[k_], yy_, color=c_, zorder=3)
    if k_ in CI: ax[0].plot(CI[k_], [yy_, yy_], color=c_)
ax[0].set_yticks(ypos); ax[0].set_yticklabels([ESTLAB[k_] for k_ in ek], fontsize=7); ax[0].set(title="(a) Main population: estimate (and 95% CI where defined)", xlabel="relative reduction (negative = support looks harmful)"); ax[0].legend(fontsize=8)
bp = ax[1].boxplot([REP[k_] for k_ in ek][::-1], vert=False, patch_artist=True, widths=.6, showfliers=True)
for p_, k_ in zip(bp["boxes"], ek[::-1]): p_.set_facecolor("#abebc6" if k_ in good else "#f5b7b1")
ax[1].axvline(TRUE_EFFECT, color="k", ls="--", lw=1.2); ax[1].set_yticklabels([ESTLAB[k_] for k_ in ek][::-1], fontsize=7)
ax[1].set(title=f"(b) Spread across {len(REP)} independent populations", xlabel="relative reduction")
ya = np.arange(len(ADJ))[::-1]; ax[2].axvline(TRUE_EFFECT, color="k", ls="--", lw=1.2)
for yy_, e_, lo_, hi_, ok_ in zip(ya, ADJ.estimate, ADJ.lo, ADJ.hi, ADJ["covers 30%"]):
    c_ = "#1e8449" if ok_ else "#c0392b"; ax[2].scatter(e_, yy_, color=c_, zorder=3); ax[2].plot([lo_, hi_], [yy_, yy_], color=c_)
ax[2].set_yticks(ya); ax[2].set_yticklabels(ADJ["adjustment set"], fontsize=8); ax[2].set(title="(c) Which signals the analyst adjusts for (green = interval covers 30%)", xlabel="Mantel–Haenszel relative reduction")
yc = np.arange(len(CC))[::-1]; ax[3].axvline(TRUE_EFFECT, color="k", ls="--", lw=1.2)
for yy_, r_ in zip(yc, CC.itertuples()): ax[3].scatter(r_.estimate, yy_, color="#2471a3", zorder=3); ax[3].plot([r_.lo, r_.hi], [yy_, yy_], color="#2471a3")
ax[3].set_yticks(yc); ax[3].set_yticklabels([f"{r_.cell} ({r_.signals} signal{'s' if r_.signals > 1 else ''})" for r_ in CC.itertuples()], fontsize=8)
ax[3].set(title="(d) Cell-specific estimates (only cells with both arms)", xlabel="relative reduction")
for a_ in ax: a_.grid(alpha=.25)
savefig(fig, "fig5_causal.png", "Does support work? Estimates against the known 30% effect"); plt.show()

**Reading the adjustment sets and cells.**
* **Adjusting for some of the referral signals is not enough, and can be misleading.** Grade alone gives −7.0 %, attendance alone −20.4 %, hold alone −37.1 %, grade plus attendance +13.6 %, grade plus hold +3.6 % (interval −2.6 % to +9.5 %), attendance plus hold −6.7 %. Only the full set of three (+32.2 %) and the count of signals fired (+32.4 %) recover the effect. The count is enough because it determines the assignment probability, and a balancing score is all that is needed (Gelman, Hill & Vehtari 2020; Peters, Janzing & Schölkopf 2017).
* **The practical lesson is about what to log.** An evaluation can only be as good as the record of the inputs to the referral decision. If an institution records the support flag but not the signals that triggered it, the programme looks harmful no matter how well it works.
* **I cannot learn effect heterogeneity from this data.** Only six cells contain both arms. All six intervals cover 30 %, but they are wide (for example attendance alone, −6.9 % to +36.2 %), and the three two-signal cells rest on only 122 to 211 unsupported student-terms. Their point estimates (41.5 % to 45.9 %) sit above 30 %. I read this as noise from thin comparison groups and not as a real difference, because the true effect is homogeneous by construction.

## 9 · Who is flagged, who is missed, and what do false flags cost?
Two views of the same programme. **Sec. 9.1 is the analyst's view**, built only from the exported tables: where the dropouts are and which of them the programme reaches. **Sec. 9.2 uses the hidden state to grade the targeting.** It asks what a support slot is actually worth, and whether a better-ranked list would buy more.

**The unit of value.** Support cuts the hazard by 30 % *for whoever receives it* (`LOCKED`, homogeneous by construction). So the expected number of dropouts averted by one support slot in a student-term is 0.30 × that student-term's *untreated* hazard. Summing over the slots a policy fills gives its expected value, with no outcome noise. This value is only as good as the assumption of a homogeneous 30 % effect, which I flag in Sec. 12.

### 9.1 The analyst's view: who is flagged and where the dropouts are

In [ ]:
FE["k"] = FE.k_l1.astype(int)
FE["group"] = np.where(FE.Tr, "supported", np.where(FE.k >= 1, "1+ signals, not supported", "clean record, not supported"))
g9 = FE.groupby("k").agg(student_terms=("y", "size"), supported=("Tr", "sum"), dropout_events=("y", "sum"))
g9["share supported"] = g9.supported / g9.student_terms; g9["dropout rate"] = g9.dropout_events / g9.student_terms; g9["share of all dropouts"] = g9.dropout_events / g9.dropout_events.sum()
banner(); print("By number of prior-term signals fired (student-terms in terms 2–8):")
display(g9.style.format({"student_terms": "{:,}", "supported": "{:,}", "dropout_events": "{:,}", "share supported": "{:.1%}", "dropout rate": "{:.1%}", "share of all dropouts": "{:.1%}"}))
ev = FE.groupby("group").agg(student_terms=("y", "size"), dropout_events=("y", "sum"))
ev["share of student-terms"] = ev.student_terms / ev.student_terms.sum(); ev["share of dropout events"] = ev.dropout_events / ev.dropout_events.sum(); ev["dropout rate"] = ev.dropout_events / ev.student_terms
print("Where the dropout events fall:"); display(ev.style.format({"student_terms": "{:,}", "dropout_events": "{:,}", "share of student-terms": "{:.1%}", "share of dropout events": "{:.1%}", "dropout rate": "{:.1%}"}))

**Where the dropouts are.** Dropout rises with the number of prior-term signals (6.8 %, 11.4 %, 12.5 %, 16.7 %), so the referral rule points in the right direction. But three-quarters of student-terms have a clean record, and they produce 63.7 % of all dropout events. Another 26.9 % of events fall in terms with at least one signal that were not supported. Only 9.4 % of dropout events occur in supported terms. The programme reaches 6.2 % of student-terms, and its rule cannot see the largest block of dropouts, which is why this is the first thing a programme manager should look at.

### 9.2 What is a support slot worth? Policies at equal capacity *(uses the hidden state to grade the targeting)*
I fix the capacity at what the actual programme uses (the number of supported student-terms in terms 2 to 8) and compare who would be chosen under five policies. Each policy's value is the sum of 0.30 × untreated hazard over the slots it fills:
1. **The actual programme:** the stochastic referral rule on prior-term signals.
2. **The boosted-tree model:** the highest out-of-fold predicted risk. (Those scores describe risk-as-currently-managed, which slightly understates the risk of students who were supported.)
3. **The rule by count:** fill slots from the highest signal count downward, breaking ties at random.
4. **The oracle:** the highest true untreated hazard. It is unattainable, since it needs the hidden baseline vulnerability, and it sets the ceiling.
5. **Random.**

The last column is the share of slots on students whose baseline vulnerability is below the cohort average, the generator's own definition of a falsely flagged student. Read it with care for the oracle, which also chases *this term's shocks* and so can legitimately pick a below-average-vulnerability student.

In [ ]:
POL = None
if GT_OK:
    h0 = h_true / (1 - TRUE_EFFECT * Trv)                       # untreated hazard (support only enters multiplicatively)
    benefit = TRUE_EFFECT * h0; total_pool = benefit.sum(); n_ = len(benefit)
    rng9 = np.random.default_rng(Q("cv_seed") + 9)
    sel = {"1. Actual programme": Trv, "2. Boosted-tree model, top capacity": top_flag}
    order_rule = np.lexsort((rng9.random(n_), -FE.k.to_numpy())); m_rule = np.zeros(n_, bool); m_rule[order_rule[:cap]] = True; sel["3. Signal-count rule, ties at random"] = m_rule
    m_or = np.zeros(n_, bool); m_or[np.argsort(-h0)[:cap]] = True; sel["4. Oracle (true untreated hazard)"] = m_or
    m_rd = np.zeros(n_, bool); m_rd[rng9.choice(n_, cap, replace=False)] = True; sel["5. Random"] = m_rd
    v_low = vb < 0
    POL = pd.DataFrame([{"policy": k_, "support slots": int(m_.sum()), "expected dropouts averted": benefit[m_].sum(),
                         "per 100 slots": 100 * benefit[m_].mean(), "share of oracle value": benefit[m_].sum() / benefit[m_or].sum(),
                         "share of all avertable dropouts": benefit[m_].sum() / total_pool, "slots on below-average baseline vulnerability": v_low[m_].mean()} for k_, m_ in sel.items()])
    banner(); print(f"Capacity = {cap:,} support slots out of {n_:,} student-terms ({cap/n_:.1%}). Total avertable if every term were supported: {total_pool:,.0f} dropouts.")
    display(POL.style.format({"support slots": "{:,}", "expected dropouts averted": "{:,.0f}", "per 100 slots": "{:.2f}", "share of oracle value": "{:.1%}", "share of all avertable dropouts": "{:.1%}",
                              "slots on below-average baseline vulnerability": "{:.1%}"}).hide(axis="index"))
    print(f"Edge cases from Sec. 4.6: {int(e1.sum()):,} dropout events ({e1.sum()/n_drop0:.1%} of all) were shock-driven, on a clean record, with no support; "
          f"{int(e2.sum()):,} supported terms ({e2.sum()/n_sup0:.1%}) were one-signal flags on below-average-vulnerability students.")
else:
    print("Skipped: exact reproduction of the exported dataset is not confirmed on this platform (see Sec. 3).")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
grp_cols = {"supported": "#1e8449", "1+ signals, not supported": "#e67e22", "clean record, not supported": "#c0392b"}
b1 = b2 = 0
for g_ in ["supported", "1+ signals, not supported", "clean record, not supported"]:
    s1, s2 = ev.loc[g_, "share of student-terms"], ev.loc[g_, "share of dropout events"]
    ax[0].bar(0, s1, bottom=b1, color=grp_cols[g_], label=g_); ax[0].bar(1, s2, bottom=b2, color=grp_cols[g_])
    ax[0].text(0, b1 + s1 / 2, f"{s1:.1%}", ha="center", va="center", fontsize=8, color="w"); ax[0].text(1, b2 + s2 / 2, f"{s2:.1%}", ha="center", va="center", fontsize=8, color="w"); b1 += s1; b2 += s2
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["student-terms", "dropout events"]); ax[0].set(title="(a) Where the dropouts are (analyst view)", ylabel="share"); ax[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=1, frameon=False)
if POL is not None:
    cs = np.arange(1, n_ + 1)
    for lab_, ordr, col_, ls_ in [("oracle", np.argsort(-h0), "#7d3c98", "-"), ("boosted-tree model", np.argsort(-sc), "#1e8449", "-"), ("signal-count rule", order_rule, "#2471a3", "-")]:
        ax[1].plot(cs / n_, np.cumsum(benefit[ordr]) / total_pool, color=col_, ls=ls_, label=lab_)
    ax[1].plot([0, 1], [0, 1], color="k", lw=.8, label="random"); ax[1].axvline(cap / n_, color="#7f8c8d", ls=":"); ax[1].set_xlim(0, 0.4); ax[1].set_ylim(0, 0.8)
    ax[1].scatter([cap / n_], [POL.iloc[0]["share of all avertable dropouts"]], color="#c0392b", zorder=4, label="actual programme")
    ax[1].set(title="(b) Expected dropouts averted by filling the first x of slots", xlabel="share of student-terms supported", ylabel="share of all avertable dropouts"); ax[1].legend(fontsize=8)
    ax[2].barh(POL.policy[::-1], POL["per 100 slots"][::-1], color=["#95a5a6", "#7d3c98", "#2471a3", "#1e8449", "#c0392b"])
    for i_, v_ in enumerate(POL["per 100 slots"][::-1]): ax[2].text(v_ + .02, i_, f"{v_:.2f}", va="center", fontsize=8)
    ax[2].set(title="(c) Expected dropouts averted per 100 support slots", xlabel="dropouts averted per 100 slots")
for a_ in ax: a_.grid(alpha=.25)
savefig(fig, "fig6_targeting.png", "Targeting audit"); plt.show()

**Reading the policy comparison.** All numbers are expected values inside the synthetic model.
* **The actual programme is worth about 796 averted dropouts** out of a total of 6,097 that could be averted if every student-term were supported. That is 13.1 % of the avertable total and 56.0 % of what an oracle with the same number of slots would achieve (1,421).
* **A model-ranked list does better at the same cost.** Filling the same 15,293 slots from the boosted-tree ranking averts about 940, which is 18 % more than the actual rule and 66.2 % of the oracle. The signal-count rule with random tie-breaking (809) is about the same as the actual rule, since the actual rule is that rule plus noise.
* **In practical terms:** about 19 support slots per averted dropout under the actual rule, about 16 under the model, and about 11 under the oracle.
* **What the last column means.** The share of slots on students with below-average baseline vulnerability is 11.9 % for the actual rule, 1.6 % for the model and 59.4 % for random allocation. The oracle's 11.0 % is not a failure of targeting. It reflects that the oracle also chases *this term's shocks*, which can hit any student. That is also why the oracle is unattainable.
* **Why I would not over-read the +18 %.** It assumes that support cuts the hazard by 30 % for everyone (`LOCKED`, homogeneous), so benefit is proportional to risk and ranking by risk is optimal by construction. If real effects vary, the best list for benefit is not the best list for risk. A pilot should test the two allocation rules against each other directly.

## 10 · How robust is all of this? Sensitivity to the `ASSUMED` parameters
Most of what I found depends on parameters the whitepaper had to assume. Two groups matter most:
* **Assignment (`β0`, `β`).** The default `(−9, 6)` is the high-precision regime the whitepaper implies. The generator notebook's open flag OF-1 kept it and deferred weaker-targeting settings to this section, because its own scan did not recalibrate. I vary it from `(−7, 6)` down to `(−2, 1)`, where even clean-record terms are sometimes supported and all four signal counts contain both arms.
* **The data-generating process.** How often shocks occur (`shock_prob`), how strongly a shock raises the hazard (`γ2`), and how noisy grades are as a proxy for vulnerability (`grade_noise`). These set how much of dropout is foreseeable at all, so they move the prediction ceiling more than they move the causal estimate.

**Every scenario is recalibrated.** Changing a parameter shifts tier attrition, so for each scenario I re-fit the two `CALIBRATED` quantities (δ, γ0) to the *same* cited targets, with the verbatim Phase 5b loop. (If the target is not bracketed I widen the search interval, which changes only the search range, not the objective.) That keeps every scenario a dataset that matches the cited tier rates, so differences between scenarios are not just differences in attrition level. Nothing is tuned to make a check pass.

**For each scenario I report:** the recalibrated tier attrition (a check that calibration worked), the supported share and where both arms exist, the main-population estimates, the spread of the Mantel–Haenszel and IPW estimates over replicate populations, and the AUC of the analyst's signal-count rule against the two ceilings from Sec. 6.4. Replicates per scenario: see the D3 registry.

In [ ]:
@contextmanager
def override(**kv):
    saved = {k_: PARAM[k_]["value"] for k_ in kv}
    try:
        for k_, v_ in kv.items(): PARAM[k_]["value"] = v_
        yield
    finally:
        for k_, v_ in saved.items(): PARAM[k_]["value"] = v_

def calibrate_wide(d):                                          # same objective and algorithm as the verbatim loop, wider search brackets
    tier = d["tier"]
    def inner(delta):
        L = simulate_latent(d, delta)
        g0 = bisect(lambda g: apply_hazard(L, d["u_drop"], g)[1][~tier].mean(), -20, 10, P("target_typical_2022"))
        return g0, apply_hazard(L, d["u_drop"], g0)[1][tier].mean()
    delta = bisect(lambda dl: inner(dl)[1], 0.0, 12.0, P("target_high_2022"), n=34)
    return delta, inner(delta)[0]

def run_scenario(group, label, params, n_rep):
    t0 = time.time()
    with override(**params):
        d = draw_noise(N, T, [SEED, 0], noncore=False); widened = False
        try: delta, g0 = calibrate(d)
        except AssertionError: delta, g0 = calibrate_wide(d); widened = True
        with override(delta=delta, gamma0=g0):
            L = simulate_latent(d, delta); dp, at, fi, hz = apply_hazard(L, d["u_drop"], g0)
            y_, T_, c_, p_, s_, m2_ = arrays_from_engine(L, dp, fi); k_ = ((c_ >> 2) & 1) + ((c_ >> 1) & 1) + (c_ & 1)
            es = estimator_suite(y_, T_, c_, p_, s_, poisson=True)
            v_ = np.repeat(L["v"][:, None], T, axis=1)[m2_]; ps_ = P("shock_prob")
            pv_ = (1 - ps_) * sig(g0 + P("gamma1") * v_) + ps_ * sig(g0 + P("gamma1") * v_ + P("gamma2"))
            both = [int(j) for j in range(4) if (T_ & (k_ == j)).sum() >= Q("min_arm") and ((~T_) & (k_ == j)).sum() >= Q("min_arm")]
            rr_ = []
            for r_ in range(1, n_rep + 1):
                d_r = draw_noise(N, T, [SEED, r_], noncore=False); L_r = simulate_latent(d_r, delta); dp_r, _, fi_r, _ = apply_hazard(L_r, d_r["u_drop"], g0)
                a_ = arrays_from_engine(L_r, dp_r, fi_r); rr_.append(estimator_suite(a_[0], a_[1], a_[2], a_[3], a_[4], poisson=False))
            rr_ = pd.DataFrame(rr_)
            return {"group": group, "scenario": label, "delta": delta, "gamma0": g0, "widened search": widened,
                    "tier high": at[d["tier"]].mean(), "tier typical": at[~d["tier"]].mean(), "supported share": T_.mean(), "cells with both arms (k)": str(both), "arm sizes by k (supported / not)": "; ".join(f"{j}: {int((T_ & (k_ == j)).sum()):,}/{int(((~T_) & (k_ == j)).sum()):,}" for j in range(4)),
                    "naive": es["naive"], "MH": es["mh"], "Poisson": es["poisson"], "IPW Hajek": es["ipw_hajek"], "IPW HT": es["ipw_ht"],
                    "MH rep mean": rr_["mh"].mean(), "MH rep SD": rr_["mh"].std(ddof=1), "IPW Hajek rep mean": rr_["ipw_hajek"].mean(), "IPW Hajek rep SD": rr_["ipw_hajek"].std(ddof=1),
                    "naive rep mean": rr_["naive"].mean(), "AUC rule": roc_auc_score(y_, k_), "AUC ceiling B": roc_auc_score(y_, pv_), "AUC ceiling A": roc_auc_score(y_, hz[m2_]), "seconds": time.time() - t0}

SCEN = [("baseline", "baseline (β0=−9, β=6)", {}),
        ("assignment", "β0=−7, β=6", dict(beta0=-7.0, beta=6.0)), ("assignment", "β0=−5, β=3", dict(beta0=-5.0, beta=3.0)),
        ("assignment", "β0=−3, β=2", dict(beta0=-3.0, beta=2.0)), ("assignment", "β0=−2, β=1", dict(beta0=-2.0, beta=1.0)),
        ("process", "shock prob 0.04", dict(shock_prob=0.04)), ("process", "shock prob 0.16", dict(shock_prob=0.16)),
        ("process", "γ2 = 0.75", dict(gamma2=0.75)), ("process", "γ2 = 3.0", dict(gamma2=3.0)),
        ("process", "grade noise 4", dict(grade_noise=4.0)), ("process", "grade noise 16", dict(grade_noise=16.0))]
if FAST: SCEN = [SCEN[0], SCEN[3], SCEN[6]]
SENS = None
if CONFIG["RUN_SENSITIVITY"]:
    snap = {k_: PARAM[k_]["value"] for k_ in PARAM}
    out_ = []
    for g_, l_, p_ in SCEN:
        out_.append(run_scenario(g_, l_, p_, Q("n_replicates_sens"))); print(f"  done: {l_:<24s} {out_[-1]['seconds']:.0f}s")
    SENS = pd.DataFrame(out_)
    assert all(PARAM[k_]["value"] == snap[k_] for k_ in PARAM), "registry was not restored"
    base_ = SENS.iloc[0]
    print(f"Baseline scenario check: MH {base_['MH']:+.1%} (main-population value from the exports: {EM['mh']:+.1%}); delta {base_['delta']:.6f} vs recorded {META['calibrated']['delta']:.6f}")
else:
    print("RUN_SENSITIVITY = False: section skipped.")

In [ ]:
if SENS is not None:
    banner()
    a_ = SENS[["scenario", "delta", "gamma0", "tier high", "tier typical", "supported share", "cells with both arms (k)", "arm sizes by k (supported / not)", "widened search"]]
    display(a_.style.format({"delta": "{:.3f}", "gamma0": "{:.3f}", "tier high": "{:.1%}", "tier typical": "{:.1%}", "supported share": "{:.1%}"}).hide(axis="index"))
    print("Causal estimates (relative reduction; truth = 30%). 'rep' = mean and SD over replicate populations at that scenario:")
    b_ = SENS[["scenario", "naive", "MH", "Poisson", "IPW Hajek", "IPW HT", "MH rep mean", "MH rep SD", "IPW Hajek rep mean", "IPW Hajek rep SD"]]
    display(b_.style.format({c_: "{:+.1%}" for c_ in b_.columns if c_ != "scenario"}).hide(axis="index"))
    print("Predictability: AUC of the analyst's signal-count rule against the ceilings of Sec. 6.4:")
    c_ = SENS[["scenario", "AUC rule", "AUC ceiling B", "AUC ceiling A"]].assign(**{"rule gap to ceiling B": lambda x: x["AUC ceiling B"] - x["AUC rule"]})
    display(c_.style.format({k_: "{:.3f}" for k_ in c_.columns if k_ != "scenario"}).hide(axis="index"))

In [ ]:
if SENS is not None:
    fig, ax = plt.subplots(1, 2, figsize=(15, 5.6)); yy = np.arange(len(SENS))[::-1]
    ax[0].axvline(TRUE_EFFECT, color="k", ls="--", lw=1.2, label="true effect 30%")
    ax[0].errorbar(SENS["MH rep mean"], yy + 0.12, xerr=2 * SENS["MH rep SD"], fmt="o", color="#1e8449", capsize=3, label="Mantel–Haenszel (mean ± 2 SD over replicates)")
    ax[0].errorbar(SENS["IPW Hajek rep mean"], yy - 0.12, xerr=2 * SENS["IPW Hajek rep SD"], fmt="s", color="#c0392b", capsize=3, label="IPW Hajek untrimmed (mean ± 2 SD)")
    ax[0].scatter(SENS["naive rep mean"], yy, marker="x", color="#7f8c8d", label="naive (replicate mean)")
    ax[0].set_yticks(yy); ax[0].set_yticklabels(SENS["scenario"], fontsize=8); ax[0].set(title="(a) Causal estimates under each scenario", xlabel="relative reduction in dropout"); ax[0].legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, frameon=False)
    ax[1].scatter(SENS["AUC rule"], yy, color="#95a5a6", label="signal-count rule", zorder=3); ax[1].scatter(SENS["AUC ceiling B"], yy, color="#7d3c98", marker="D", label="ceiling B (knows v)", zorder=3)
    ax[1].scatter(SENS["AUC ceiling A"], yy, color="#7d3c98", marker="*", s=90, label="ceiling A (true hazard)", zorder=3)
    for yy_, a_, b_ in zip(yy, SENS["AUC rule"], SENS["AUC ceiling A"]): ax[1].plot([a_, b_], [yy_, yy_], color="#d7bde2", zorder=1)
    ax[1].set_yticks(yy); ax[1].set_yticklabels(SENS["scenario"], fontsize=8); ax[1].set(title="(b) How predictable is dropout under each scenario?", xlabel="AUC"); ax[1].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)
    for a_ in ax: a_.grid(alpha=.25)
    savefig(fig, "fig7_sensitivity.png", "Sensitivity to the assumed parameters (each scenario recalibrated to the same targets)"); plt.show()

**Reading the sensitivity grid.** Every scenario recalibrates to exactly 67.1 % and 45.9 %, and none needed the widened search interval, so each is a legitimate dataset for the cited targets. The recalibrated δ runs from 1.010 to 1.191 and γ0 from −3.049 to −2.549.
* **The assignment rule drives how hard the causal problem is.** As the rule softens from (−9, 6) to (−2, 1), the supported share grows from 6.2 % to 17.0 %. The signal counts with at least 100 student-terms in each arm grow from {1, 2} to {0, 1, 2} at (−5, 3) and (−3, 2), and to all four at (−2, 1). (At (−3, 2) the three-signal group has 99 unsupported student-terms, one short of my threshold, so the generator notebook's expectation that all four cells are populated there holds in the loose sense but not by my criterion.) On the main population the naive estimate moves from −57.7 % to +12.1 %: confounding fades as assignment becomes less deterministic, but even at (−2, 1) the naive estimate understates the 30 % by 18 points.
* **Mantel–Haenszel is stable everywhere.** Main-population values run from 30.9 % to 35.0 %, and replicate means from 26.9 % to 30.1 %. Its replicate SD shrinks from 5.0 points at the default rule to 1.0 point at (−2, 1), the benefit of overlap.
* **IPW's instability is a positivity problem, and it goes away when positivity is restored.** The Hajek replicate SD falls from 35.6 points (default) to 19.5 (−7, 6; two-signal cell still saturated), 5.6 (−5, 3), 2.3 (−3, 2) and 1.1 (−2, 1). Once every signal cell contains both arms IPW is as good as Mantel–Haenszel.
* **The data-generating process matters less for the causal estimate.** Varying the shock probability, the shock hazard slope γ2 and the grade noise leaves Mantel–Haenszel within replicate noise of 30 % in every case, while IPW stays erratic (replicate SDs of 36 to 51 points).
* **Predictability is capped by baseline vulnerability, not by shocks.** The signal-count rule's AUC stays between 0.556 and 0.570 in every scenario. Ceiling B (knowing baseline vulnerability) moves only between 0.659 and 0.686. Ceiling A (the true hazard) ranges widely, from 0.696 when shocks are weak (γ2 = 0.75) to 0.800 when they are strong (γ2 = 3.0). So the part of dropout an analyst can ever hope to predict is stable across my assumptions, while how much is hidden depends strongly on how large shocks are.
* **Not varied, and so not tested:** the 30 % effect itself and the tier targets (both `LOCKED` or `CALIBRATED`), independence of shocks from prior records, interactions between parameters, and the boosted model's AUC outside the baseline. Replicates per scenario are 10, so the SDs are order-of-magnitude figures.

## 11 · Recommendations
These are recommendations about **how to build and evaluate a retention programme**, drawn from how the synthetic dataset behaves. They are not claims about any real institution, and the numbers are properties of the generator. Each is tied to the evidence for it.

1. **Use the risk model to triage, never to judge the programme** (Sec. 6, Sec. 7). In this dataset it improves on the signal-count rule (top-10 % capture of 21.4 % against 15.3 %) but its ceiling is modest: 0.722 AUC even with perfect knowledge of the hazard, and 19.4 % of dropouts occur before any record exists. Report the AUC with its fold SD, the capture at the programme's real capacity, and calibration. Where the ceiling is unknown (as it would be with real data), say so.
2. **Evaluate the programme with a design that uses the assignment rule, and log its inputs at decision time** (Sec. 7, Sec. 8). Record the signals that triggered each referral together with the support flag. Analyse with Mantel–Haenszel or a cluster-robust Poisson model, stratified on those signals. Expect a standard error of about 4 points at a cohort of 50,000, so a single evaluation resolves the effect only to about ±8 points. Do not read the raw supported-versus-unsupported comparison, and do not stratify on a risk score instead of the assignment inputs.
3. **Build overlap into the rollout** (Sec. 8, Sec. 10). With the default step-like rule, no signal-free student is ever supported and no student with three signals is ever left out, so IPW is unreliable and cell-level estimates are noisy. Softer assignment (for example, β0 = −3, β = 2) put both arms in the signal counts 0, 1 and 2 (the three-signal group had 99 unsupported student-terms) and brought the replicate SD of Mantel–Haenszel down to 1.5 points, with IPW stable too (SD 2.3 points). A small randomised share of referrals is the simplest way to get that. I did not measure the cost of this trade-off in averted dropouts per slot, so it is the first thing I would test.
4. **Allocate slots from a model-based ranking, as a pilot** (Sec. 9). At equal capacity the boosted-tree ranking averted an expected 18 % more dropouts than the actual rule (940 against 796). That comparison assumes a homogeneous effect, so I would test the two allocation rules head to head before trusting it.
5. **Look where the dropouts are, not only where the flags are** (Sec. 9.1). Most dropout events (63.7 %) occur in terms with a clean record and no support. A programme that evaluates only its own referrals will never see them.
6. **Keep regional tier out of the flagging model** (Sec. 6.3). Excluding it cost no accuracy here (AUC 0.655 against 0.656), and it avoids flagging students for support because of where they come from (WP Sec. 13).
7. **Do not rely on IPW as the first-line estimator when assignment is steep** (Sec. 8). If it is used, report the effective sample size and the largest weights, and show the answer under both normalisations and with and without trimming, because here the answer depended on them.
8. **Do not act on any number from this notebook** (all sections). These are results about a model calibrated to aggregate statistics. The point of the exercise is the method: validate a data source against known truth, separate prediction from causal evaluation, and test how much each conclusion depends on what was assumed.

## 12 · Limitations, open flags and the D3 assumptions log
Statuses follow the generator notebook: **OPEN** = unresolved, carried forward · **NOTED** = disclosed with a stated consequence · **RESOLVED** = decided here.

**Carried forward from the generator notebook (Deliverable 2)**

| ID | Flag | Status in this deliverable |
|---|---|---|
| OF-1 | Realized targeting does not match the Section 8 design targets (sensitivity 70 %, false-flag share 25 %). The generator kept (β0, β) = (−9, 6) and reserved weaker-targeting settings for this deliverable. | **RESOLVED** in the generator; the deferred part is done in Sec. 10: four weaker-assignment scenarios, each recalibrated to the cited tier targets. |
| OF-2 | The held-out tier ratio sits at the band's upper edge; the single-seed pass is fragile (about 60 % of replicate populations in band). | **NOTED.** I did not re-run the held-out check. Anything calibrated through a one-parameter level shift inherits this limit. |
| OF-7 | The 29.4 % national figure is read as a Class-of-2023 value. | **OPEN** (inherited from the whitepaper). |
| OF-9 | CHED "attrition" includes temporary leave; the model has no re-entry, so rates are an upper bound on permanent exit. | **NOTED.** |
| OF-10 | The Mantel–Haenszel interval treats student-terms as independent; the Poisson estimator clusters by student. | **NOTED.** I report both again (Sec. 4.5, Sec. 8). |
| OF-12 | IPW results depend on the implementation. | **NOTED**, and extended: Sec. 8 shows the dependence across 20 populations, and Sec. 10 shows it vanishes when overlap is restored. |

**New in this deliverable**

| ID | Flag | Status |
|---|---|---|
| DF-1 | **Everything here is a property of the generator.** Predictability ceilings, the homogeneous 30 % effect, the form of confounding and the absence of unmeasured assignment confounders are all assumptions of the model. None is a measurement. | **NOTED.** The banner on every output says so. |
| DF-2 | **Homogeneous effect and linear policy value.** Sec. 9.2 values a slot as 0.30 times the untreated hazard, so ranking by risk is optimal by construction. With heterogeneous real effects it need not be. | **NOTED.** Recommendation 4 asks for a head-to-head pilot. |
| DF-3 | **No carry-over.** In the generator, support changes the hazard but never later grades, attendance or holds, so there is no time-varying confounding. A real programme would create it. Assignment also depends only on observed signals, with no caseworker discretion. | **NOTED.** |
| DF-4 | **The analyst is assumed to know the referral thresholds** (`ASSUMED`) **and to see the true propensity** in `risk_score`. The IPW results are therefore a best case. | **NOTED.** |
| DF-5 | **Model comparison is narrow.** Two model families, fixed hyperparameters (not tuned), one fold assignment, one population. The fold SD does not include population-draw variability, and the model's AUC was not recomputed across the Sec. 10 scenarios. | **NOTED.** |
| DF-6 | **Sensitivity is one parameter at a time**, with 10 replicates per scenario. Interactions are untested, and SDs are rough. | **NOTED.** |
| DF-7 | **Mechanism of the stratify-on-score bias** (Sec. 7) is stated as likely and not investigated. | **OPEN.** |
| DF-8 | **Term-1 dropouts** (19.4 % of all dropouts) are outside both the prediction and the causal analysis, because support is never assigned in term 1 and no prior record exists. | **NOTED.** |
| DF-9 | **Citations.** Methods are cited by author and year as in my whitepaper's reference list. The Lopez-Muñoz et al. (2026) review was confirmed by web search while writing this notebook (*Journal of Information Technology Education: Research*, vol. 25, https://doi.org/10.28945/5776). | **RESOLVED.** |

**Decision principle (unchanged from Deliverable 2).** I do not change a calibrated or pre-specified quantity to make a result look better. In the sensitivity grid, the only quantities re-fitted are the two `CALIBRATED` ones, and they are re-fitted to the same cited targets.

### D3 assumptions log (auto-generated from the D3 registry)

In [ ]:
a3 = pd.DataFrame([{"parameter": k_, "value": (json.dumps(v_["value"]) if isinstance(v_["value"], (dict, tuple, list)) else v_["value"]), "tag": v_["tag"], "basis": v_["basis"]} for k_, v_ in A3.items()])
assert a3.tag.isin({"LOCKED", "CALIBRATED", "ASSUMED"}).all()
print(f"{len(a3)} D3 choices, each with exactly one tag: {a3.tag.value_counts().to_dict()}. The generator's own {len(PARAM)} parameters are logged in parameter_log.csv.")
display(a3.style.hide(axis="index").set_properties(subset=["basis"], **{"text-align": "left"}))

## 13 · Reproducibility
* **Run order.** `Run All` from the top. The notebook reads the exported CSV files, so Deliverable 2 does not need to be re-run first. It finds the data from either the repository root or `results-and-discussion/`.
* **Run time.** About 10 to 15 minutes on a laptop in full mode. `FAST_MODE` in Sec. 1 gives a quick smoke test (not comparable with the scorecard); `RUN_SENSITIVITY = False` skips Sec. 10.
* **Randomness.** Every random step is seeded (the generator's master seed 193, replicate seeds `[193, r]`, a fixed fold seed). Results are bitwise reproducible on the same platform and library versions, and statistically equivalent elsewhere (WP App. B.3 #5).
* **If the hash check in Sec. 3 fails on your platform,** the notebook says so, continues, and skips only the steps that need the exports and the regenerated latent state to line up row by row.
* **Dependencies** are pinned in `requirements.txt` in this folder.
* **Figures** are written to `figures/` by the notebook; the repository also holds an executed HTML copy so the results can be read without running anything.

In [ ]:
print("Environment for this run")
print(f"  python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__} | statsmodels {statsmodels.__version__} | scikit-learn {sklearn.__version__}")
print(f"  platform: {platform.platform()}")
print(f"  dataset: generator {META['generator_version']}, N = {N:,}, seed = {SEED}; generated on {META['versions']['platform']} with numpy {META['versions']['numpy']}, pandas {META['versions']['pandas']}")
print(f"  exact reproduction of the dataset by the engine copy on this platform: {PARITY_EXACT}")
print(f"  FAST_MODE = {FAST}; RUN_SENSITIVITY = {CONFIG['RUN_SENSITIVITY']}")
print("  figures written to:", sorted(p.name for p in FIG.glob('*.png')))
banner()